# ImageGrains 2.0: Resumable Full-Dataset Ground-Truth Validation Suite

This notebook evaluates **every unique ground-truth photograph (238 examples)** across **ImageGrains 2.0** (`Zenodo 17866827`), **ImageGrains v1** (`Zenodo 8005771`), and **`dmair1989/imagegrains`**—comparing Original PyTorch (`IG2_full_set.200525` & `IG2_full_set_cp_SAM`), `ONNX + Cellpose Dynamics`, and our ablation-optimized **Browser Engine v4 (`F5_MagWeightedQC + F3_SplitOnFailure`)**.

### Architecture & Crash-Proof Resume Design
1. **Clean Separation of Inference vs. Morphometrics:** `venv_cp3` (`cellpose==3.1.1.1`) and `venv_cp4` (`cellpose>=4.0.9,<4.2`) run strictly model inference (`PyTorch` + `ONNX`) using the exact dependency set verified in `v11`, while the main Colab Python environment runs `Browser Engine v4`, `regionprops`, and `matplotlib` plotting—preventing any NumPy/SciPy version collisions.
2. **Atomic Per-Image State Tracking (`state/unet/{uid}.json` & `state/cpsam/{uid}.json`):** Every image writes an atomic JSON record to Google Drive immediately upon completion. If Colab disconnects mid-batch, **already-completed images are automatically skipped**.
3. **Batched Drive Report Checkpointing (`BATCH_SIZE = 20`):** Updates `full_dataset_benchmark.csv`, `full_dataset_validation_report.md`, and `figures/aggregate_d50_parity.png` on Google Drive after every 20 images.

## Step 1: Mount Google Drive, Check Hardware & Setup Isolated Inference Envs

In [ ]:
import glob, os, shutil, subprocess, psutil, torch

DRIVE_ROOT = '/content/drive/MyDrive/imagegrains_full_validation'
DRIVE_DATA_DIR = '/content/drive/MyDrive/imagegrains_datasets'
DRIVE_MODEL_DIR = os.path.join(DRIVE_DATA_DIR, 'models')

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

for d in [
    '/content/weights',
    '/content/onnx',
    '/content/test_images',
    '/content/batch_tmp/unet',
    '/content/batch_tmp/cpsam',
    DRIVE_ROOT,
    os.path.join(DRIVE_ROOT, 'state', 'unet'),
    os.path.join(DRIVE_ROOT, 'state', 'cpsam'),
    os.path.join(DRIVE_ROOT, 'figures'),
    DRIVE_DATA_DIR,
    DRIVE_MODEL_DIR,
]:
    os.makedirs(d, exist_ok=True)

ram_gb = psutil.virtual_memory().total / (1024**3)
disk_gb = shutil.disk_usage('/content').free / (1024**3)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU mode)'
print(f'RAM: {ram_gb:.1f} GB | Free Disk: {disk_gb:.1f} GB | GPU: {gpu_name}')
print(f'Persistent Google Drive Checkpoint Dir: {DRIVE_ROOT}')

subprocess.run(['pip', 'install', '-q', 'uv', 'pandas', 'matplotlib', 'scikit-image', 'scipy', 'opencv-python-headless'], check=True)

def verify_venv(py_bin: str, label: str) -> bool:
    if not os.path.exists(py_bin):
        return False
    res = subprocess.run(
        [py_bin, '-c', 'import numpy, torch, onnxruntime, cv2; from cellpose import models, dynamics'],
        capture_output=True, text=True
    )
    if res.returncode != 0:
        print(f'[{label}] Verification notice:\n{res.stderr.strip()}')
        return False
    return True

# Use the exact minimal dependency set proven in v11 (no extra scipy/skimage/matplotlib pins inside venvs)
if not verify_venv('/content/venv_cp3/bin/python', 'venv_cp3'):
    print('Building venv_cp3 (cellpose==3.1.1.1)...')
    subprocess.run(['uv', 'venv', '/content/venv_cp3', '--python', '3.11', '--clear', '-q'], check=True)
    subprocess.run([
        'uv', 'pip', 'install', '--python', '/content/venv_cp3/bin/python', '-q',
        'cellpose==3.1.1.1', 'torch', 'onnxruntime-gpu', 'opencv-python-headless', 'numpy'
    ], check=True)
    if not verify_venv('/content/venv_cp3/bin/python', 'venv_cp3'):
        raise RuntimeError('Failed to verify /content/venv_cp3 imports (see stderr printed above).')

if not verify_venv('/content/venv_cp4/bin/python', 'venv_cp4'):
    print('Building venv_cp4 (cellpose>=4.0.9,<4.2)...')
    subprocess.run(['uv', 'venv', '/content/venv_cp4', '--python', '3.11', '--clear', '-q'], check=True)
    subprocess.run([
        'uv', 'pip', 'install', '--python', '/content/venv_cp4/bin/python', '-q',
        'cellpose>=4.0.9,<4.2', 'torch', 'onnxruntime-gpu', 'opencv-python-headless', 'numpy'
    ], check=True)
    if not verify_venv('/content/venv_cp4/bin/python', 'venv_cp4'):
        raise RuntimeError('Failed to verify /content/venv_cp4 imports (see stderr printed above).')

print('Both isolated inference environments (venv_cp3 & venv_cp4) verified and ready.')

## Step 2: Build Deduplicated Master Manifest of ALL 238 Ground-Truth Examples

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, urllib.request, zipfile
from PIL import Image

DRIVE_ROOT = '/content/drive/MyDrive/imagegrains_full_validation'
DRIVE_DATA_DIR = '/content/drive/MyDrive/imagegrains_datasets'
MASTER_MANIFEST_DRIVE = os.path.join(DRIVE_ROOT, 'full_dataset_manifest.json')
MASTER_MANIFEST_LOCAL = '/content/test_images/full_dataset_manifest.json'

INCLUDE_IG1_V1_ARCHIVE = True  # Includes ImageGrains v1 (Zenodo 8005771) alongside IG2_data.zip & GitHub repo

# 1. Ensure IG2_data.zip (911.6 MB) is cached on Drive and extracted locally
drive_ig2_zip = os.path.join(DRIVE_DATA_DIR, 'IG2_data.zip')
local_ig2_zip = '/content/test_images/IG2_data.zip'

if not os.path.exists(drive_ig2_zip) and os.path.exists('/content/drive/MyDrive'):
    matches = glob.glob('/content/drive/MyDrive/**/*IG2_data*.zip', recursive=True)
    if matches:
        drive_ig2_zip = matches[0]

if os.path.exists(drive_ig2_zip) and os.path.getsize(drive_ig2_zip) > 100_000_000:
    print(f'Found cached IG2_data.zip on Drive: {drive_ig2_zip} ({os.path.getsize(drive_ig2_zip)/(1024**2):.1f} MB)')
    if not os.path.exists(local_ig2_zip):
        shutil.copy2(drive_ig2_zip, local_ig2_zip)
elif os.path.exists(local_ig2_zip) and os.path.getsize(local_ig2_zip) > 100_000_000:
    print(f'Found local IG2_data.zip: {local_ig2_zip}')
    shutil.copy2(local_ig2_zip, drive_ig2_zip)
else:
    url = 'https://zenodo.org/records/17866827/files/IG2_data.zip?download=1'
    print('Downloading IG2_data.zip from Zenodo (911.6 MB) and saving to Drive...')
    urllib.request.urlretrieve(url, local_ig2_zip)
    shutil.copy2(local_ig2_zip, drive_ig2_zip)

ig2_extract_dir = '/content/test_images/extracted/IG2_data'
if not os.path.exists(ig2_extract_dir) or len(os.listdir(ig2_extract_dir)) == 0:
    os.makedirs(ig2_extract_dir, exist_ok=True)
    print('Extracting IG2_data.zip...')
    with zipfile.ZipFile(local_ig2_zip, 'r') as zf:
        zf.extractall(ig2_extract_dir)

# 2. Optionally cache & extract ImageGrains v1 dataset (Zenodo 8005771)
v1_extract_dir = '/content/test_images/extracted/ImageGrains_v1_data'
if INCLUDE_IG1_V1_ARCHIVE:
    drive_v1_zip = os.path.join(DRIVE_DATA_DIR, 'ImageGrains_v1_data.zip')
    local_v1_zip = '/content/test_images/ImageGrains_v1_data.zip'
    if not os.path.exists(drive_v1_zip) and os.path.exists('/content/drive/MyDrive'):
        matches_v1 = glob.glob('/content/drive/MyDrive/**/*ImageGrains_v1_data*.zip', recursive=True)
        if matches_v1:
            drive_v1_zip = matches_v1[0]
    if os.path.exists(drive_v1_zip) and os.path.getsize(drive_v1_zip) > 10_000_000:
        if not os.path.exists(local_v1_zip):
            shutil.copy2(drive_v1_zip, local_v1_zip)
    elif not os.path.exists(local_v1_zip):
        try:
            req = urllib.request.urlopen('https://zenodo.org/api/records/8005771', timeout=20)
            meta = json.loads(req.read().decode('utf-8'))
            for f_info in meta.get('files', []):
                if f_info.get('key', '').endswith('.zip'):
                    print(f"Downloading Zenodo 8005771/{f_info['key']} ({f_info['size']/(1024**2):.1f} MB)...")
                    urllib.request.urlretrieve(f_info['links']['self'], local_v1_zip)
                    shutil.copy2(local_v1_zip, drive_v1_zip)
                    break
        except Exception as e:
            print(f'Zenodo 8005771 optional download skipped: {e}')
    if os.path.exists(local_v1_zip) and (not os.path.exists(v1_extract_dir) or len(os.listdir(v1_extract_dir)) == 0):
        os.makedirs(v1_extract_dir, exist_ok=True)
        with zipfile.ZipFile(local_v1_zip, 'r') as zf:
            zf.extractall(v1_extract_dir)

# 3. Clone official GitHub repository
if not os.path.exists('/content/imagegrains_repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q', 'https://github.com/dmair1989/imagegrains.git', '/content/imagegrains_repo'], check=True)

# 4. Scan ALL trees, match every (image, mask) pair, and deduplicate by MD5 image hash
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
MASK_SUFFIXES = ('_mask', '_masks', '_seg', '_gt', '_label', '_labels', '_cp_masks')
EXCLUDE_IMG_KEYWORDS = ('logo', 'fig', 'overview', 'workflow', 'banner', 'flow', 'pred')

def is_mask_file(path: str) -> bool:
    stem = os.path.splitext(os.path.basename(path))[0].lower()
    parent = os.path.basename(os.path.dirname(path)).lower()
    return any(stem.endswith(s) for s in MASK_SUFFIXES) or parent in ('masks', 'mask', 'gt', 'labels', 'annotations')

def find_matching_mask(img_path: str, all_files: list[str]) -> str | None:
    stem = os.path.splitext(os.path.basename(img_path))[0].lower()
    img_dir = os.path.dirname(img_path)
    parent_dir = os.path.dirname(img_dir)

    for f in all_files:
        if f == img_path:
            continue
        f_dir = os.path.dirname(f)
        if f_dir != img_dir and os.path.dirname(f_dir) != parent_dir:
            continue
        f_stem = os.path.splitext(os.path.basename(f))[0].lower()
        if any(f_stem == stem + s for s in MASK_SUFFIXES):
            return f
        if os.path.basename(f_dir).lower() in ('masks', 'mask', 'gt', 'labels', 'annotations') and f_stem == stem:
            return f

    for f in all_files:
        if f == img_path:
            continue
        f_stem = os.path.splitext(os.path.basename(f))[0].lower()
        if any(f_stem == stem + s for s in MASK_SUFFIXES):
            return f
        if os.path.basename(os.path.dirname(f)).lower() in ('masks', 'mask', 'gt', 'labels', 'annotations') and f_stem == stem:
            return f
    return None

def file_md5(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def collect_all_pairs(root_dir: str, source_label: str) -> list[dict]:
    if not os.path.exists(root_dir):
        return []
    all_files = [
        p for p in glob.glob(f'{root_dir}/**/*', recursive=True)
        if os.path.isfile(p) and p.lower().endswith(IMG_EXTS + ('.npy',))
    ]
    found = []
    for p in sorted(all_files):
        if not p.lower().endswith(IMG_EXTS) or is_mask_file(p):
            continue
        low = os.path.basename(p).lower()
        if any(k in low for k in EXCLUDE_IMG_KEYWORDS):
            continue
        m_path = find_matching_mask(p, all_files)
        if m_path is None:
            continue
        try:
            with Image.open(p) as im:
                w, h = im.size
                if w < 128 or h < 128:
                    continue
        except Exception:
            continue
        split_name = 'test' if 'test' in p.lower() else ('val' if 'val' in p.lower() else 'train')
        site_bucket = os.path.basename(os.path.dirname(p))
        found.append({
            'orig_img': p,
            'orig_mask': m_path,
            'source': source_label,
            'split': split_name,
            'site': site_bucket,
            'width': w,
            'height': h,
            'pixels': w * h,
            'md5': file_md5(p),
        })
    return found

raw_pairs = []
raw_pairs.extend(collect_all_pairs('/content/imagegrains_repo', 'github:dmair1989/imagegrains'))
raw_pairs.extend(collect_all_pairs(ig2_extract_dir, 'zenodo:17866827/IG2_data.zip'))
if INCLUDE_IG1_V1_ARCHIVE:
    raw_pairs.extend(collect_all_pairs(v1_extract_dir, 'zenodo:8005771/ImageGrains_v1_data.zip'))

# Sort so official 'test' split entries take priority during MD5 deduplication, then smaller-to-larger images
raw_pairs.sort(key=lambda x: (0 if x['split'] == 'test' else (1 if x['split'] == 'val' else 2), x['pixels'], os.path.basename(x['orig_img'])))

unique_records = []
seen_md5 = set()
for item in raw_pairs:
    if item['md5'] in seen_md5:
        continue
    seen_md5.add(item['md5'])
    base_stem = os.path.splitext(os.path.basename(item['orig_img']))[0]
    uid = f"{item['md5'][:8]}_{base_stem}"
    unique_records.append({
        'uid': uid,
        'md5': item['md5'],
        'image_path': item['orig_img'],
        'mask_path': item['orig_mask'],
        'source': item['source'],
        'split': item['split'],
        'site': item['site'],
        'width': item['width'],
        'height': item['height'],
    })

with open(MASTER_MANIFEST_LOCAL, 'w') as f:
    json.dump(unique_records, f, indent=2)
shutil.copy2(MASTER_MANIFEST_LOCAL, MASTER_MANIFEST_DRIVE)

test_n = sum(1 for r in unique_records if r['split'] == 'test')
train_n = len(unique_records) - test_n
print(f'Discovered {len(raw_pairs)} total paired files -> {len(unique_records)} UNIQUE ground-truth examples (test={test_n}, train/val={train_n}).')
print(f'Master manifest saved to: {MASTER_MANIFEST_DRIVE}')

## Step 3: Load or Cache All 4 Checkpoints & ONNX Models on Google Drive

In [ ]:
import os, shutil, urllib.request
from tqdm.auto import tqdm

DRIVE_MODEL_DIR = '/content/drive/MyDrive/imagegrains_datasets/models'

DOWNLOADS = [
    (
        'https://zenodo.org/records/15728186/files/IG2_full_set.200525?download=1',
        '/content/weights/IG2_full_set.200525',
    ),
    (
        'https://zenodo.org/records/15728186/files/IG2_full_set_cp_SAM?download=1',
        '/content/weights/IG2_full_set_cp_SAM',
    ),
    (
        'https://huggingface.co/inYourOwnBrowser/imagegrains-2.0-onnx/resolve/main/ig2_unet_fp32.onnx',
        '/content/onnx/ig2_unet_fp32.onnx',
    ),
    (
        'https://huggingface.co/inYourOwnBrowser/imagegrains-2.0-onnx/resolve/main/ig2_cpsam_fp16.onnx',
        '/content/onnx/ig2_cpsam_fp16.onnx',
    ),
]

def fetch_with_drive_cache(url, dest):
    bname = os.path.basename(dest)
    drive_copy = os.path.join(DRIVE_MODEL_DIR, bname)

    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f'Cached locally: {dest} ({os.path.getsize(dest)/(1024*1024):.1f} MB)')
        if not os.path.exists(drive_copy):
            shutil.copy2(dest, drive_copy)
        return

    if os.path.exists(drive_copy) and os.path.getsize(drive_copy) > 0:
        shutil.copy2(drive_copy, dest)
        print(f'Restored from Google Drive cache: {dest} ({os.path.getsize(dest)/(1024*1024):.1f} MB)')
        return

    req = urllib.request.urlopen(url)
    total = int(req.headers.get('Content-Length', 0))
    with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=bname) as pbar:
        while True:
            chunk = req.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            pbar.update(len(chunk))

    shutil.copy2(dest, drive_copy)
    print(f'Saved model to Drive cache: {drive_copy}')

for url, dest in DOWNLOADS:
    fetch_with_drive_cache(url, dest)

## Step 4: Write Minimal Inference Workers (`infer_unet.py` & `infer_cpsam.py`)

In [ ]:
%%writefile /content/infer_unet.py
import argparse, inspect, json, os, time
import cv2
import numpy as np
import onnxruntime as ort
import torch
from cellpose import dynamics, models


def load_rgb_image(img_path: str) -> np.ndarray:
    img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img_bgr is not None:
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    from PIL import Image
    return np.array(Image.open(img_path).convert("RGB"))


def load_instance_mask(mask_path: str, expected_hw: tuple[int, int]) -> np.ndarray:
    if not mask_path or not os.path.exists(mask_path):
        return np.zeros(expected_hw, dtype=np.int32)
    arr = np.load(mask_path) if mask_path.endswith(".npy") else cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if arr is None:
        return np.zeros(expected_hw, dtype=np.int32)
    if arr.ndim == 3:
        if np.array_equal(arr[:, :, 0], arr[:, :, 1]) and np.array_equal(arr[:, :, 0], arr[:, :, 2]):
            arr = arr[:, :, 0]
        else:
            rgb = arr[:, :, :3].astype(np.int64)
            arr = (rgb[:, :, 0] << 16) | (rgb[:, :, 1] << 8) | rgb[:, :, 2]
    if arr.shape[:2] != expected_hw:
        arr = cv2.resize(arr.astype(np.int32), (expected_hw[1], expected_hw[0]), interpolation=cv2.INTER_NEAREST)
    arr_i = arr.astype(np.int64)
    arr_i[arr_i < 0] = 0
    uniq, inv = np.unique(arr_i, return_inverse=True)
    remapped = inv.reshape(expected_hw).astype(np.int32)
    if len(uniq) > 0 and uniq[0] != 0:
        remapped += 1
    return remapped


def normalize_gray(img_rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    p1, p99 = np.percentile(gray, [1, 99])
    return ((gray - p1) / (p99 - p1 + 1e-8)).astype(np.float32)


def run_onnx_unet_full(sess: ort.InferenceSession, norm_gray: np.ndarray) -> np.ndarray:
    H, W = norm_gray.shape
    pad_h = (16 - (H % 16)) % 16
    pad_w = (16 - (W % 16)) % 16
    padded = norm_gray if (pad_h == 0 and pad_w == 0) else np.pad(norm_gray, ((0, pad_h), (0, pad_w)), mode="reflect")
    inp = np.zeros((1, 2, padded.shape[0], padded.shape[1]), dtype=np.float32)
    inp[0, 0] = padded
    return sess.run(["flows_cellprob"], {"image": inp})[0][0][:, :H, :W]


def reconstruct_masks_compat(dP: np.ndarray, cellprob: np.ndarray, device):
    sig = inspect.signature(dynamics.compute_masks)
    kwargs = {"niter": 200, "cellprob_threshold": 0.0, "flow_threshold": 0.4, "min_size": 30}
    if "interp" in sig.parameters:
        kwargs["interp"] = True
    if "device" in sig.parameters:
        kwargs["device"] = device
    res = dynamics.compute_masks(dP, cellprob, **kwargs)
    return res[0] if isinstance(res, tuple) else res


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--batch-file", required=True)
    parser.add_argument("--state-dir", required=True)
    parser.add_argument("--tmp-dir", required=True)
    args = parser.parse_args()

    os.makedirs(args.tmp_dir, exist_ok=True)
    with open(args.batch_file) as f:
        batch_items = json.load(f)

    pending = [it for it in batch_items if not os.path.exists(os.path.join(args.state_dir, f"{it['uid']}.json"))]
    if not pending:
        return

    ckpt_path = "/content/weights/IG2_full_set.200525"
    onnx_path = "/content/onnx/ig2_unet_fp32.onnx"

    use_gpu = torch.cuda.is_available()
    cp_model = models.CellposeModel(gpu=use_gpu, pretrained_model=ckpt_path)
    ref_net = cp_model.net.float().eval()

    cpu_sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    ort_sess = cpu_sess
    if use_gpu:
        try:
            cuda_sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
            probe = np.random.RandomState(0).rand(1, 2, 256, 256).astype(np.float32)
            if float(np.max(np.abs(cpu_sess.run(["flows_cellprob"], {"image": probe})[0] - cuda_sess.run(["flows_cellprob"], {"image": probe})[0]))) < 1e-3:
                ort_sess = cuda_sess
        except Exception:
            pass

    _ = run_onnx_unet_full(ort_sess, np.zeros((256, 256), dtype=np.float32))

    for i, item in enumerate(pending):
        uid = item["uid"]
        tmp_npz = os.path.join(args.tmp_dir, f"{uid}.npz")
        if os.path.exists(tmp_npz):
            continue

        img_rgb = load_rgb_image(item["image_path"])
        H, W = img_rgb.shape[:2]
        norm_gray = normalize_gray(img_rgb)
        gt_masks = load_instance_mask(item["mask_path"], (H, W))

        cy, cx = max(0, H // 2 - 128), max(0, W // 2 - 128)
        crop = norm_gray[cy:cy + 256, cx:cx + 256]
        if crop.shape != (256, 256):
            crop = cv2.resize(norm_gray, (256, 256), interpolation=cv2.INTER_LINEAR)
        pt_inp = torch.zeros((1, 2, 256, 256), dtype=torch.float32, device=cp_model.device)
        pt_inp[0, 0] = torch.from_numpy(crop).to(device=cp_model.device, dtype=torch.float32)
        with torch.no_grad():
            pt_raw = ref_net(pt_inp)[0][0].cpu().numpy()
        onnx_raw = ort_sess.run(["flows_cellprob"], {"image": pt_inp.cpu().numpy()})[0][0]
        raw_max_err = float(np.max(np.abs(pt_raw - onnx_raw)))
        raw_mae = float(np.mean(np.abs(pt_raw - onnx_raw)))

        t0 = time.perf_counter()
        pt_masks = cp_model.eval(
            img_rgb, channels=[0, 0], diameter=30.0,
            flow_threshold=0.4, cellprob_threshold=0.0, min_size=30, normalize=True
        )[0].astype(np.int32)
        pt_sec = time.perf_counter() - t0

        t0 = time.perf_counter()
        onnx_out = run_onnx_unet_full(ort_sess, norm_gray)
        onnx_inf_sec = time.perf_counter() - t0
        dP, cellprob = onnx_out[:2], onnx_out[2]

        t0 = time.perf_counter()
        cp_masks = reconstruct_masks_compat(dP, cellprob, cp_model.device).astype(np.int32)
        cp_post_sec = time.perf_counter() - t0

        np.savez(
            tmp_npz,
            img_rgb=img_rgb,
            gt_masks=gt_masks,
            pt_masks=pt_masks,
            cp_masks=cp_masks,
            dP=dP.astype(np.float32),
            cellprob=cellprob.astype(np.float32),
            raw_max_err=np.array(raw_max_err),
            raw_mae=np.array(raw_mae),
            pt_sec=np.array(pt_sec),
            onnx_inf_sec=np.array(onnx_inf_sec),
            cp_post_sec=np.array(cp_post_sec),
        )
        print(f"  [U-Net Infer {i+1}/{len(pending)}] {uid} ({H}x{W}) | PT={pt_masks.max()} | ONNX+CP={cp_masks.max()}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/infer_cpsam.py
import argparse, inspect, json, os, time
import cv2
import numpy as np
import onnxruntime as ort
import torch
from cellpose import dynamics, models


def load_rgb_image(img_path: str) -> np.ndarray:
    img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img_bgr is not None:
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    from PIL import Image
    return np.array(Image.open(img_path).convert("RGB"))


def load_instance_mask(mask_path: str, expected_hw: tuple[int, int]) -> np.ndarray:
    if not mask_path or not os.path.exists(mask_path):
        return np.zeros(expected_hw, dtype=np.int32)
    arr = np.load(mask_path) if mask_path.endswith(".npy") else cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if arr is None:
        return np.zeros(expected_hw, dtype=np.int32)
    if arr.ndim == 3:
        if np.array_equal(arr[:, :, 0], arr[:, :, 1]) and np.array_equal(arr[:, :, 0], arr[:, :, 2]):
            arr = arr[:, :, 0]
        else:
            rgb = arr[:, :, :3].astype(np.int64)
            arr = (rgb[:, :, 0] << 16) | (rgb[:, :, 1] << 8) | rgb[:, :, 2]
    if arr.shape[:2] != expected_hw:
        arr = cv2.resize(arr.astype(np.int32), (expected_hw[1], expected_hw[0]), interpolation=cv2.INTER_NEAREST)
    arr_i = arr.astype(np.int64)
    arr_i[arr_i < 0] = 0
    uniq, inv = np.unique(arr_i, return_inverse=True)
    remapped = inv.reshape(expected_hw).astype(np.int32)
    if len(uniq) > 0 and uniq[0] != 0:
        remapped += 1
    return remapped


def normalize_rgb_joint99(img_rgb: np.ndarray) -> np.ndarray:
    arr = img_rgb.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    return ((arr - p1) / (p99 - p1 + 1e-8)).astype(np.float32)


def make_blend_window(tile_size: int = 256) -> np.ndarray:
    ramp = np.hanning(tile_size).astype(np.float32)
    ramp = np.clip(ramp, 1e-3, 1.0)
    return np.outer(ramp, ramp)


def _tile_starts(length: int, tile_size: int, stride: int) -> list[int]:
    if length <= tile_size:
        return [0]
    starts = list(range(0, length - tile_size, stride))
    if not starts or starts[-1] != length - tile_size:
        starts.append(length - tile_size)
    return starts


def run_onnx_cpsam_tiled(
    sess: ort.InferenceSession,
    norm_rgb: np.ndarray,
    tile_size: int = 256,
    overlap: int = 64,
    batch_size: int = 16,
) -> np.ndarray:
    H, W, _ = norm_rgb.shape
    if H == tile_size and W == tile_size:
        batch_arr = np.transpose(norm_rgb, (2, 0, 1))[None].astype(np.float16)
        return sess.run(["flows_cellprob"], {"image": batch_arr})[0][0].astype(np.float32)

    pad_h = max(0, tile_size - H)
    pad_w = max(0, tile_size - W)
    work_img = np.pad(norm_rgb, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect") if (pad_h > 0 or pad_w > 0) else norm_rgb

    Hw, Ww, _ = work_img.shape
    stride = tile_size - overlap
    ys = _tile_starts(Hw, tile_size, stride)
    xs = _tile_starts(Ww, tile_size, stride)
    coords = [(y, x) for y in ys for x in xs]

    accum = np.zeros((3, Hw, Ww), dtype=np.float32)
    weights = np.zeros((Hw, Ww), dtype=np.float32)
    win = make_blend_window(tile_size)

    for start in range(0, len(coords), batch_size):
        batch_coords = coords[start:start + batch_size]
        batch_arr = np.empty((len(batch_coords), 3, tile_size, tile_size), dtype=np.float16)
        for i, (y, x) in enumerate(batch_coords):
            batch_arr[i] = np.transpose(work_img[y:y + tile_size, x:x + tile_size], (2, 0, 1)).astype(np.float16)
        preds = sess.run(["flows_cellprob"], {"image": batch_arr})[0].astype(np.float32)
        for i, (y, x) in enumerate(batch_coords):
            accum[:, y:y + tile_size, x:x + tile_size] += preds[i] * win[None, :, :]
            weights[y:y + tile_size, x:x + tile_size] += win

    return (accum / np.maximum(weights[None, :, :], 1e-6))[:, :H, :W]


def reconstruct_masks_compat(dP: np.ndarray, cellprob: np.ndarray, device):
    sig = inspect.signature(dynamics.compute_masks)
    kwargs = {"niter": 200, "cellprob_threshold": 0.0, "flow_threshold": 0.4, "min_size": 30}
    if "interp" in sig.parameters:
        kwargs["interp"] = True
    if "device" in sig.parameters:
        kwargs["device"] = device
    res = dynamics.compute_masks(dP, cellprob, **kwargs)
    return res[0] if isinstance(res, tuple) else res


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--batch-file", required=True)
    parser.add_argument("--state-dir", required=True)
    parser.add_argument("--tmp-dir", required=True)
    args = parser.parse_args()

    os.makedirs(args.tmp_dir, exist_ok=True)
    with open(args.batch_file) as f:
        batch_items = json.load(f)

    pending = [it for it in batch_items if not os.path.exists(os.path.join(args.state_dir, f"{it['uid']}.json"))]
    if not pending:
        return

    ckpt_path = "/content/weights/IG2_full_set_cp_SAM"
    onnx_path = "/content/onnx/ig2_cpsam_fp16.onnx"

    use_gpu = torch.cuda.is_available()
    cp_model = models.CellposeModel(gpu=use_gpu, pretrained_model=ckpt_path)

    ort_sess = None
    if use_gpu:
        try:
            ort_sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        except Exception as e:
            print(f"CUDAExecutionProvider warning ({e}); falling back to CPUExecutionProvider")
    if ort_sess is None:
        ort_sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

    _ = ort_sess.run(["flows_cellprob"], {"image": np.zeros((1, 3, 256, 256), dtype=np.float16)})

    for i, item in enumerate(pending):
        uid = item["uid"]
        tmp_npz = os.path.join(args.tmp_dir, f"{uid}.npz")
        if os.path.exists(tmp_npz):
            continue

        img_rgb = load_rgb_image(item["image_path"])
        H, W = img_rgb.shape[:2]
        norm_rgb = normalize_rgb_joint99(img_rgb)
        gt_masks = load_instance_mask(item["mask_path"], (H, W))

        cy, cx = max(0, H // 2 - 128), max(0, W // 2 - 128)
        crop = norm_rgb[cy:cy + 256, cx:cx + 256]
        if crop.shape[:2] != (256, 256):
            crop = cv2.resize(norm_rgb, (256, 256), interpolation=cv2.INTER_LINEAR)
        crop_chw = np.transpose(crop, (2, 0, 1))[None]
        pt_inp = torch.from_numpy(crop_chw).to(device=cp_model.device, dtype=torch.float32)
        orig_dtype = cp_model.net.dtype
        cp_model.net.dtype = torch.float32
        with torch.no_grad():
            pt_raw = cp_model.net.float()(pt_inp)[0][0].cpu().numpy()
        cp_model.net.dtype = orig_dtype
        if orig_dtype != torch.float32:
            cp_model.net.to(orig_dtype)

        onnx_raw = ort_sess.run(["flows_cellprob"], {"image": crop_chw.astype(np.float16)})[0][0].astype(np.float32)
        raw_max_err = float(np.max(np.abs(pt_raw - onnx_raw)))
        raw_mae = float(np.mean(np.abs(pt_raw - onnx_raw)))

        t0 = time.perf_counter()
        pt_masks = cp_model.eval(
            img_rgb, diameter=None, flow_threshold=0.4, cellprob_threshold=0.0,
            min_size=30, tile_overlap=0.25, normalize=True
        )[0].astype(np.int32)
        pt_sec = time.perf_counter() - t0

        t0 = time.perf_counter()
        onnx_out = run_onnx_cpsam_tiled(ort_sess, norm_rgb, tile_size=256, overlap=64, batch_size=16)
        onnx_inf_sec = time.perf_counter() - t0
        dP, cellprob = onnx_out[:2], onnx_out[2]

        t0 = time.perf_counter()
        cp_masks = reconstruct_masks_compat(dP, cellprob, cp_model.device).astype(np.int32)
        cp_post_sec = time.perf_counter() - t0

        np.savez(
            tmp_npz,
            img_rgb=img_rgb,
            gt_masks=gt_masks,
            pt_masks=pt_masks,
            cp_masks=cp_masks,
            dP=dP.astype(np.float32),
            cellprob=cellprob.astype(np.float32),
            raw_max_err=np.array(raw_max_err),
            raw_mae=np.array(raw_mae),
            pt_sec=np.array(pt_sec),
            onnx_inf_sec=np.array(onnx_inf_sec),
            cp_post_sec=np.array(cp_post_sec),
        )
        print(f"  [CP-SAM Infer {i+1}/{len(pending)}] {uid} ({H}x{W}) | PT={pt_masks.max()} | ONNX+CP={cp_masks.max()}")


if __name__ == "__main__":
    main()

## Step 5: Write Main-Kernel Browser Engine v4 & Morphometrics Core (`eval_core.py`)

In [ ]:
%%writefile /content/eval_core.py
import collections, datetime, glob, json, os, platform, time, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage
from skimage.measure import regionprops


def _dilate_binary_3x3(mask: np.ndarray, iterations: int = 2) -> np.ndarray:
    out = mask.astype(bool)
    for _ in range(iterations):
        p = np.pad(out, 1, mode='constant', constant_values=False)
        out = (
            p[:-2, :-2] | p[:-2, 1:-1] | p[:-2, 2:] |
            p[1:-1, :-2] | p[1:-1, 1:-1] | p[1:-1, 2:] |
            p[2:, :-2] | p[2:, 1:-1] | p[2:, 2:]
        )
    return out


def _label_connected_8(binary_mask: np.ndarray) -> tuple[np.ndarray, int]:
    labels, n = ndimage.label(binary_mask, structure=np.ones((3, 3), dtype=np.int32))
    return labels.astype(np.int32), int(n)


def _run_jacobi_grid(sub_mask: np.ndarray, T: np.ndarray, cy_c: int, cx_c: int, steps: int) -> np.ndarray:
    for _ in range(steps):
        T[cy_c, cx_c] += 1.0
        T[1:-1, 1:-1] = np.where(
            sub_mask,
            (
                T[:-2, :-2] + T[:-2, 1:-1] + T[:-2, 2:] +
                T[1:-1, :-2] + T[1:-1, 1:-1] + T[1:-1, 2:] +
                T[2:, :-2] + T[2:, 1:-1] + T[2:, 2:]
            ) * (1.0 / 9.0),
            0.0,
        )
    return T


def _multigrid_jacobi_heat(sub_mask: np.ndarray, cy: int, cx: int) -> np.ndarray:
    bh, bw = sub_mask.shape
    max_dim = max(bh, bw)
    T = np.zeros((bh + 2, bw + 2), dtype=np.float32)

    if max_dim >= 192:
        m8 = sub_mask[::8, ::8]
        h8, w8 = m8.shape
        T8 = _run_jacobi_grid(
            m8,
            np.zeros((h8 + 2, w8 + 2), dtype=np.float32),
            min(cy // 8, h8 - 1) + 1,
            min(cx // 8, w8 - 1) + 1,
            max(60, 2 * (h8 + w8)),
        )
        T_up = np.repeat(np.repeat(T8[1:-1, 1:-1], 8, axis=0), 8, axis=1)[:bh, :bw]
        T[1:-1, 1:-1] = np.where(sub_mask, T_up, 0.0)
        n_fine = min(120, max(60, (bh + bw) // 2))
    elif max_dim >= 96:
        m4 = sub_mask[::4, ::4]
        h4, w4 = m4.shape
        T4 = _run_jacobi_grid(
            m4,
            np.zeros((h4 + 2, w4 + 2), dtype=np.float32),
            min(cy // 4, h4 - 1) + 1,
            min(cx // 4, w4 - 1) + 1,
            max(60, 2 * (h4 + w4)),
        )
        T_up = np.repeat(np.repeat(T4[1:-1, 1:-1], 4, axis=0), 4, axis=1)[:bh, :bw]
        T[1:-1, 1:-1] = np.where(sub_mask, T_up, 0.0)
        n_fine = min(120, max(60, (bh + bw) // 2))
    elif max_dim >= 48:
        m2 = sub_mask[::2, ::2]
        h2, w2 = m2.shape
        T2 = _run_jacobi_grid(
            m2,
            np.zeros((h2 + 2, w2 + 2), dtype=np.float32),
            min(cy // 2, h2 - 1) + 1,
            min(cx // 2, w2 - 1) + 1,
            max(60, 2 * (h2 + w2)),
        )
        T_up = np.repeat(np.repeat(T2[1:-1, 1:-1], 2, axis=0), 2, axis=1)[:bh, :bw]
        T[1:-1, 1:-1] = np.where(sub_mask, T_up, 0.0)
        n_fine = min(120, max(60, (bh + bw) // 2))
    else:
        n_fine = max(50, 2 * (bh + bw))

    return _run_jacobi_grid(sub_mask, T, cy + 1, cx + 1, n_fine)


def _evaluate_grain_flow_qc_v4(
    sub_mask: np.ndarray,
    sub_uy: np.ndarray,
    sub_ux: np.ndarray,
    sub_mag: np.ndarray,
    flow_thresh: float,
) -> bool:
    my, mx = np.nonzero(sub_mask)
    if len(my) == 0:
        return False
    mean_y = float(np.mean(my))
    mean_x = float(np.mean(mx))
    center_idx = int(np.argmin((my - mean_y) ** 2 + (mx - mean_x) ** 2))
    cy_c = int(my[center_idx])
    cx_c = int(mx[center_idx])

    T = _multigrid_jacobi_heat(sub_mask, cy_c, cx_c)
    gy = T[2:, 1:-1] - T[:-2, 1:-1]
    gx = T[1:-1, 2:] - T[1:-1, :-2]
    gmag = np.sqrt(gy * gy + gx * gx) + 1e-6
    ideal_uy = gy / gmag
    ideal_ux = gx / gmag

    err_sq = (5.0 * sub_uy[sub_mask] - 5.0 * ideal_uy[sub_mask]) ** 2 + (5.0 * sub_ux[sub_mask] - 5.0 * ideal_ux[sub_mask]) ** 2

    dist2_center = (my - cy_c) ** 2 + (mx - cx_c) ** 2
    w_mag = np.minimum(1.0, sub_mag[sub_mask] / 1.5)
    w_mag = np.where(dist2_center <= 2, 0.0, w_mag)
    w_sum = float(np.sum(w_mag))
    flow_err = float(np.sum(err_sq * w_mag) / max(w_sum, 1e-6))

    return (flow_err / 10.0) <= flow_thresh


def _try_split_dumbbell_v4(
    sub_mask: np.ndarray,
    sub_uy: np.ndarray,
    sub_ux: np.ndarray,
    sub_mag: np.ndarray,
    sub_sink_hist: np.ndarray,
    min_size: int,
    flow_thresh: float,
) -> list[np.ndarray]:
    bh, bw = sub_mask.shape
    my, mx = np.nonzero(sub_mask)
    if len(my) < 2 * min_size or (bh < 2 and bw < 2):
        return []

    gy_u = np.gradient(sub_uy, axis=0) if bh >= 2 else np.zeros_like(sub_uy)
    gx_u = np.gradient(sub_ux, axis=1) if bw >= 2 else np.zeros_like(sub_ux)
    inward_div = np.maximum(0.0, -(gy_u + gx_u))
    score = np.where(sub_mask, sub_sink_hist.astype(np.float32) + 2.0 * inward_div, -1.0)

    p1_flat = int(np.argmax(score))
    y1, x1 = divmod(p1_flat, bw)

    yy, xx = np.ogrid[:bh, :bw]
    score_2 = np.where(((yy - y1) ** 2 + (xx - x1) ** 2) >= 36, score, -1.0)
    p2_flat = int(np.argmax(score_2))
    if score_2.ravel()[p2_flat] <= 0:
        return []
    y2, x2 = divmod(p2_flat, bw)

    part = np.zeros((bh, bw), dtype=np.int32)
    part[y1, x1] = 1
    part[y2, x2] = 2
    q = collections.deque([(y1, x1, 1), (y2, x2, 2)])
    while q:
        cy, cx, cid = q.popleft()
        for ny in range(max(0, cy - 1), min(bh, cy + 2)):
            for nx in range(max(0, cx - 1), min(bw, cx + 2)):
                if sub_mask[ny, nx] and part[ny, nx] == 0:
                    part[ny, nx] = cid
                    q.append((ny, nx, cid))

    rem_y, rem_x = np.nonzero(sub_mask & (part == 0))
    if len(rem_y) > 0:
        d1 = (rem_y - y1) ** 2 + (rem_x - x1) ** 2
        d2 = (rem_y - y2) ** 2 + (rem_x - x2) ** 2
        part[rem_y, rem_x] = np.where(d1 <= d2, 1, 2)

    valid_children = []
    for cid in (1, 2):
        child = (part == cid)
        if int(child.sum()) < min_size:
            continue
        if _evaluate_grain_flow_qc_v4(child, sub_uy, sub_ux, sub_mag, flow_thresh):
            valid_children.append(child)
    return valid_children


def browser_engine_v4_flows_to_masks(
    dP: np.ndarray,
    cellprob: np.ndarray,
    niter: int = 250,
    cellprob_thresh: float = 0.0,
    flow_thresh: float = 0.45,
    min_size: int = 30,
    min_seed_density: int = 2,
) -> np.ndarray:
    H, W = cellprob.shape
    fg = cellprob > cellprob_thresh
    ys, xs = np.nonzero(fg)
    if len(ys) == 0:
        return np.zeros((H, W), dtype=np.int32)

    raw_mag = np.sqrt(dP[0] * dP[0] + dP[1] * dP[1])
    mag = raw_mag + 1e-6
    uy = np.where(fg, dP[0] / mag, 0.0).astype(np.float32)
    ux = np.where(fg, dP[1] / mag, 0.0).astype(np.float32)

    py = ys.astype(np.float32)
    px = xs.astype(np.float32)
    h_max = float(H - 1)
    w_max = float(W - 1)

    coarse_steps = min(100, niter // 2)
    coarse_stride = max(1.0, max(H, W) / 256.0)

    for step in range(niter):
        step_scale = coarse_stride if step < coarse_steps else 1.0
        y0 = np.floor(py).astype(np.int32)
        x0 = np.floor(px).astype(np.int32)
        y1 = np.minimum(y0 + 1, H - 1)
        x1 = np.minimum(x0 + 1, W - 1)
        wy = py - y0
        wx = px - x0

        w00 = (1.0 - wy) * (1.0 - wx)
        w01 = (1.0 - wy) * wx
        w10 = wy * (1.0 - wx)
        w11 = wy * wx

        dy = w00 * uy[y0, x0] + w01 * uy[y0, x1] + w10 * uy[y1, x0] + w11 * uy[y1, x1]
        dx = w00 * ux[y0, x0] + w01 * ux[y0, x1] + w10 * ux[y1, x0] + w11 * ux[y1, x1]

        py = np.clip(py + step_scale * dy, 0.0, h_max)
        px = np.clip(px + step_scale * dx, 0.0, w_max)

    sy = np.clip(np.rint(py).astype(np.int32), 0, H - 1)
    sx = np.clip(np.rint(px).astype(np.int32), 0, W - 1)
    sink_hist = np.zeros((H, W), dtype=np.int32)
    np.add.at(sink_hist, (sy, sx), 1)

    seed_mask = sink_hist >= min_seed_density
    if not np.any(seed_mask):
        seed_mask = sink_hist > 0
    dilated_seeds = _dilate_binary_3x3(seed_mask, iterations=2)
    seed_labels, n_seeds = _label_connected_8(dilated_seeds)
    if n_seeds == 0:
        return np.zeros((H, W), dtype=np.int32)

    assigned = seed_labels[sy, sx]
    unassigned = np.where(assigned == 0)[0]
    if len(unassigned) > 0:
        pad_seeds = np.pad(seed_labels, 2, mode='constant', constant_values=0)
        for idx_p in unassigned:
            yy = int(sy[idx_p]) + 2
            xx = int(sx[idx_p]) + 2
            patch = pad_seeds[yy - 2:yy + 3, xx - 2:xx + 3]
            nz = patch[patch > 0]
            if len(nz) > 0:
                assigned[idx_p] = int(nz[0])

    raw_labels = np.zeros((H, W), dtype=np.int32)
    raw_labels[ys, xs] = assigned

    counts = np.bincount(raw_labels.ravel())
    valid_ids = np.where(counts >= min_size)[0]
    valid_ids = valid_ids[valid_ids > 0]
    if len(valid_ids) == 0:
        return np.zeros((H, W), dtype=np.int32)

    is_valid_lut = np.zeros(len(counts), dtype=bool)
    is_valid_lut[valid_ids] = True
    fg_valid = is_valid_lut[raw_labels]

    vy, vx = np.nonzero(fg_valid)
    vlabels = raw_labels[vy, vx]
    max_lbl = int(valid_ids.max())
    y_min = np.full(max_lbl + 1, H, dtype=np.int32)
    y_max = np.full(max_lbl + 1, -1, dtype=np.int32)
    x_min = np.full(max_lbl + 1, W, dtype=np.int32)
    x_max = np.full(max_lbl + 1, -1, dtype=np.int32)
    np.minimum.at(y_min, vlabels, vy)
    np.maximum.at(y_max, vlabels, vy)
    np.minimum.at(x_min, vlabels, vx)
    np.maximum.at(x_max, vlabels, vx)

    final_masks = np.zeros((H, W), dtype=np.int32)
    next_id = 1

    for mid in valid_ids:
        y0_b, y1_b = int(y_min[mid]), int(y_max[mid]) + 1
        x0_b, x1_b = int(x_min[mid]), int(x_max[mid]) + 1
        sub_mask = (raw_labels[y0_b:y1_b, x0_b:x1_b] == mid)
        if int(sub_mask.sum()) < min_size:
            continue

        sub_uy = uy[y0_b:y1_b, x0_b:x1_b]
        sub_ux = ux[y0_b:y1_b, x0_b:x1_b]
        sub_mag = raw_mag[y0_b:y1_b, x0_b:x1_b]

        if _evaluate_grain_flow_qc_v4(sub_mask, sub_uy, sub_ux, sub_mag, flow_thresh):
            roi = final_masks[y0_b:y1_b, x0_b:x1_b]
            roi[sub_mask & (roi == 0)] = next_id
            next_id += 1
        else:
            children = _try_split_dumbbell_v4(
                sub_mask,
                sub_uy,
                sub_ux,
                sub_mag,
                sink_hist[y0_b:y1_b, x0_b:x1_b],
                min_size,
                flow_thresh,
            )
            for child in children:
                roi = final_masks[y0_b:y1_b, x0_b:x1_b]
                roi[child & (roi == 0)] = next_id
                next_id += 1

    return final_masks


def filter_border_grains(masks: np.ndarray, margin: int = 2) -> np.ndarray:
    if masks.max() == 0:
        return masks
    border_ids = set(np.unique(masks[:margin, :]))
    border_ids.update(np.unique(masks[-margin:, :]))
    border_ids.update(np.unique(masks[:, :margin]))
    border_ids.update(np.unique(masks[:, -margin:]))
    border_ids.discard(0)
    if not border_ids:
        return masks
    out = masks.copy()
    for bid in border_ids:
        out[out == bid] = 0
    uniq = np.unique(out)
    uniq = uniq[uniq > 0]
    if len(uniq) == 0:
        return np.zeros_like(masks, dtype=np.int32)
    remap = np.zeros(int(out.max()) + 1, dtype=np.int32)
    remap[uniq] = np.arange(1, len(uniq) + 1, dtype=np.int32)
    return remap[out]


def extract_interior_psd(masks: np.ndarray, min_area: int = 30) -> tuple[dict, np.ndarray]:
    clean = filter_border_grains(masks, margin=2)
    props = regionprops(clean)
    areas = []
    b_axes = []
    for p in props:
        if p.axis_minor_length > 0 and p.area >= min_area:
            areas.append(float(p.area))
            b_axes.append(float(p.axis_minor_length))
    if not areas:
        return {
            'count': 0, 'D16': np.nan, 'D50': np.nan, 'D84': np.nan,
            'vals': np.array([]), 'cdf': np.array([])
        }, clean
    vals = np.array(b_axes, dtype=np.float64)
    wts = np.array(areas, dtype=np.float64)
    order = np.argsort(vals)
    v, w = vals[order], wts[order]
    cdf = (np.cumsum(w) - 0.5 * w) / np.sum(w)
    res = {f'D{p}': float(np.interp(p / 100.0, cdf, v)) for p in (16, 50, 84)}
    res['count'] = len(v)
    res['vals'] = v
    res['cdf'] = cdf
    return res, clean


def compute_pq(ref_masks: np.ndarray, pred_masks: np.ndarray, iou_thresh: float = 0.5) -> float:
    ref_ids = np.unique(ref_masks)
    ref_ids = ref_ids[ref_ids > 0]
    pred_ids = np.unique(pred_masks)
    pred_ids = pred_ids[pred_ids > 0]
    if len(ref_ids) == 0 and len(pred_ids) == 0:
        return 1.0
    if len(ref_ids) == 0 or len(pred_ids) == 0:
        return 0.0

    ref_areas = np.bincount(ref_masks.ravel())
    pred_areas = np.bincount(pred_masks.ravel())
    stride = int(pred_masks.max()) + 1
    pairs = ref_masks.ravel().astype(np.int64) * stride + pred_masks.ravel().astype(np.int64)
    uniq_pairs, counts = np.unique(pairs, return_counts=True)
    g_idx = uniq_pairs // stride
    p_idx = uniq_pairs % stride
    valid = (g_idx > 0) & (p_idx > 0)
    g_idx, p_idx, inter = g_idx[valid], p_idx[valid], counts[valid]
    union = ref_areas[g_idx] + pred_areas[p_idx] - inter
    ious = inter / np.maximum(union, 1)

    matched = ious >= iou_thresh
    tp = int(matched.sum())
    fp = int(len(pred_ids) - tp)
    fn = int(len(ref_ids) - tp)
    return float(ious[matched].sum() / (tp + 0.5 * fp + 0.5 * fn)) if (tp + fp + fn) > 0 else 0.0


def colorize_masks(img_rgb: np.ndarray, masks: np.ndarray, alpha: float = 0.48) -> np.ndarray:
    rng = np.random.default_rng(42)
    max_id = int(masks.max())
    palette = np.zeros((max_id + 1, 3), dtype=np.float32)
    if max_id > 0:
        palette[1:] = rng.integers(40, 255, size=(max_id, 3)).astype(np.float32)
    overlay = palette[masks]
    fg = (masks > 0)[:, :, None]
    blended = np.where(fg, (1.0 - alpha) * img_rgb.astype(np.float32) + alpha * overlay, img_rgb.astype(np.float32))
    return np.clip(blended, 0, 255).astype(np.uint8)


def save_comparison_figure(
    fig_path: str,
    title_label: str,
    uid: str,
    shape_str: str,
    img_rgb: np.ndarray,
    gt_clean: np.ndarray,
    pt_clean: np.ndarray,
    v4_clean: np.ndarray,
    gt_psd: dict,
    pt_psd: dict,
    cp_psd: dict,
    v4_psd: dict,
    pq_gt_v4: float,
    pq_cp_v4: float,
    d50_err_gt: float,
    d50_err_pt: float,
):
    fig, axes = plt.subplots(1, 5, figsize=(25, 5.0), dpi=120)
    fig.suptitle(
        f'{title_label}  |  {uid} ({shape_str})  |  PQ(BrowserV4 vs GT)={pq_gt_v4:.3f} | PQ(BrowserV4 vs ONNX+CP)={pq_cp_v4:.3f}',
        fontsize=12.5, fontweight='bold'
    )

    axes[0].imshow(img_rgb)
    axes[0].set_title('1. Input Photograph')
    axes[0].axis('off')

    axes[1].imshow(colorize_masks(img_rgb, gt_clean))
    axes[1].set_title(f"2. Human Ground Truth ({gt_psd['count']} interior)")
    axes[1].axis('off')

    axes[2].imshow(colorize_masks(img_rgb, pt_clean))
    axes[2].set_title(f"3. Original PyTorch ({pt_psd['count']} interior)")
    axes[2].axis('off')

    axes[3].imshow(colorize_masks(img_rgb, v4_clean))
    axes[3].set_title(f"4. ONNX + BrowserV4 ({v4_psd['count']} interior)")
    axes[3].axis('off')

    has_any_curve = any(d['count'] > 0 for d in (gt_psd, pt_psd, cp_psd, v4_psd))
    if gt_psd['count'] > 0:
        axes[4].plot(gt_psd['vals'], gt_psd['cdf'] * 100, label=f"Human GT (D50={gt_psd['D50']:.1f} px)", lw=2.8, color='#111111')
    if pt_psd['count'] > 0:
        axes[4].plot(pt_psd['vals'], pt_psd['cdf'] * 100, label=f"PyTorch (D50={pt_psd['D50']:.1f} px)", lw=2.2, color='#1f77b4')
    if cp_psd['count'] > 0:
        axes[4].plot(cp_psd['vals'], cp_psd['cdf'] * 100, '--', label=f"ONNX+CP (D50={cp_psd['D50']:.1f} px)", lw=2.0, color='#ff7f0e')
    if v4_psd['count'] > 0:
        axes[4].plot(v4_psd['vals'], v4_psd['cdf'] * 100, ':', label=f"ONNX+BrowserV4 (D50={v4_psd['D50']:.1f} px)", lw=2.4, color='#2ca02c')

    if has_any_curve:
        axes[4].set_xscale('log')
        axes[4].legend(loc='lower right', fontsize=8.0)
    axes[4].set_ylim(0, 100)
    axes[4].grid(True, which='both', alpha=0.3)
    axes[4].set_xlabel('Grain Intermediate Axis b (px, log scale)')
    axes[4].set_ylabel('Area-Weighted Cumulative Finer (%)')
    sub_str = f"ΔD50 vs GT={d50_err_gt:.1f}% | vs PT={d50_err_pt:.1f}%" if not np.isnan(d50_err_gt) else f"ΔD50 vs PT={d50_err_pt:.1f}%"
    axes[4].set_title(f"5. Area-Weighted PSD\n({sub_str})")

    plt.tight_layout()
    tmp_fig = fig_path + '.tmp.png'
    plt.savefig(tmp_fig, bbox_inches='tight')
    plt.close(fig)
    os.replace(tmp_fig, fig_path)


def _pct_err(pred, ref):
    return round(100.0 * abs(pred - ref) / max(ref, 1e-6), 2) if (not np.isnan(ref) and not np.isnan(pred)) else None


def _signed_pct_err(pred, ref):
    return round(100.0 * (pred - ref) / max(ref, 1e-6), 2) if (not np.isnan(ref) and not np.isnan(pred)) else None


def finalize_batch_to_drive(
    batch_items: list[dict],
    family: str,
    model_label: str,
    tmp_dir: str,
    state_dir: str,
    fig_dir: str,
    save_figures: bool,
):
    '''Runs Browser Engine v4, computes PSD & PQ, saves PNGs, and writes atomic state JSONs to Drive.'''
    for item in batch_items:
        uid = item['uid']
        out_json = os.path.join(state_dir, f'{uid}.json')
        tmp_npz = os.path.join(tmp_dir, f'{uid}.npz')
        if os.path.exists(out_json) or not os.path.exists(tmp_npz):
            continue

        data = np.load(tmp_npz)
        img_rgb = data['img_rgb']
        H, W = img_rgb.shape[:2]
        gt_masks = data['gt_masks']
        pt_masks = data['pt_masks']
        cp_masks = data['cp_masks']
        dP = data['dP']
        cellprob = data['cellprob']

        t0 = time.perf_counter()
        v4_masks = browser_engine_v4_flows_to_masks(
            dP, cellprob, niter=250, cellprob_thresh=0.0, flow_thresh=0.45, min_size=30
        )
        v4_post_sec = time.perf_counter() - t0

        gt_psd, gt_clean = extract_interior_psd(gt_masks)
        pt_psd, pt_clean = extract_interior_psd(pt_masks)
        cp_psd, _ = extract_interior_psd(cp_masks)
        v4_psd, v4_clean = extract_interior_psd(v4_masks)

        has_gt = gt_psd['count'] > 0
        pq_pt_cp = round(compute_pq(pt_masks, cp_masks), 4)
        pq_pt_v4 = round(compute_pq(pt_masks, v4_masks), 4)
        pq_cp_v4 = round(compute_pq(cp_masks, v4_masks), 4)
        pq_gt_pt = round(compute_pq(gt_masks, pt_masks), 4)
        pq_gt_cp = round(compute_pq(gt_masks, cp_masks), 4)
        pq_gt_v4 = round(compute_pq(gt_masks, v4_masks), 4)

        fig_rel = f'figures/{family}_{uid}.png'
        if save_figures:
            save_comparison_figure(
                os.path.join(fig_dir, f'{family}_{uid}.png'),
                model_label,
                uid, f'{H}x{W}', img_rgb, gt_clean, pt_clean, v4_clean,
                gt_psd, pt_psd, cp_psd, v4_psd,
                pq_gt_v4, pq_cp_v4,
                _pct_err(v4_psd['D50'], gt_psd['D50']) or np.nan,
                _pct_err(v4_psd['D50'], pt_psd['D50']) or np.nan,
            )

        rec = {
            'family': family,
            'model': model_label,
            'uid': uid,
            'split': item['split'],
            'site': item['site'],
            'source': item['source'],
            'shape': f'{H}x{W}',
            'has_gt': has_gt,
            'raw_max_err': float(data['raw_max_err']),
            'raw_mae': float(data['raw_mae']),
            'gt_grains': gt_psd['count'],
            'pt_grains': pt_psd['count'],
            'cp_grains': cp_psd['count'],
            'v4_grains': v4_psd['count'],
            'pq_cp_vs_pt': pq_pt_cp,
            'pq_v4_vs_pt': pq_pt_v4,
            'pq_v4_vs_cp': pq_cp_v4,
            'pq_pt_vs_gt': pq_gt_pt,
            'pq_cp_vs_gt': pq_gt_cp,
            'pq_v4_vs_gt': pq_gt_v4,
            'gt_D16': round(gt_psd['D16'], 2) if has_gt else None,
            'gt_D50': round(gt_psd['D50'], 2) if has_gt else None,
            'gt_D84': round(gt_psd['D84'], 2) if has_gt else None,
            'pt_D16': round(pt_psd['D16'], 2) if pt_psd['count'] > 0 else None,
            'pt_D50': round(pt_psd['D50'], 2) if pt_psd['count'] > 0 else None,
            'pt_D84': round(pt_psd['D84'], 2) if pt_psd['count'] > 0 else None,
            'cp_D16': round(cp_psd['D16'], 2) if cp_psd['count'] > 0 else None,
            'cp_D50': round(cp_psd['D50'], 2) if cp_psd['count'] > 0 else None,
            'cp_D84': round(cp_psd['D84'], 2) if cp_psd['count'] > 0 else None,
            'v4_D16': round(v4_psd['D16'], 2) if v4_psd['count'] > 0 else None,
            'v4_D50': round(v4_psd['D50'], 2) if v4_psd['count'] > 0 else None,
            'v4_D84': round(v4_psd['D84'], 2) if v4_psd['count'] > 0 else None,
            'pt_D16_err_gt': _pct_err(pt_psd['D16'], gt_psd['D16']),
            'pt_D50_err_gt': _pct_err(pt_psd['D50'], gt_psd['D50']),
            'pt_D84_err_gt': _pct_err(pt_psd['D84'], gt_psd['D84']),
            'cp_D16_err_gt': _pct_err(cp_psd['D16'], gt_psd['D16']),
            'cp_D50_err_gt': _pct_err(cp_psd['D50'], gt_psd['D50']),
            'cp_D84_err_gt': _pct_err(cp_psd['D84'], gt_psd['D84']),
            'v4_D16_err_gt': _pct_err(v4_psd['D16'], gt_psd['D16']),
            'v4_D50_err_gt': _pct_err(v4_psd['D50'], gt_psd['D50']),
            'v4_D84_err_gt': _pct_err(v4_psd['D84'], gt_psd['D84']),
            'v4_D50_bias_gt': _signed_pct_err(v4_psd['D50'], gt_psd['D50']),
            'cp_D50_err_pt': _pct_err(cp_psd['D50'], pt_psd['D50']),
            'v4_D50_err_pt': _pct_err(v4_psd['D50'], pt_psd['D50']),
            'pt_sec': round(float(data['pt_sec']), 3),
            'onnx_inf_sec': round(float(data['onnx_inf_sec']), 3),
            'cp_post_sec': round(float(data['cp_post_sec']), 3),
            'v4_post_sec': round(v4_post_sec, 3),
            'figure_path': fig_rel,
        }

        tmp_json = out_json + '.tmp'
        with open(tmp_json, 'w') as f:
            json.dump(rec, f, indent=2)
        os.replace(tmp_json, out_json)

        os.remove(tmp_npz)
        print(
            f"  [{family.upper()} Saved] {uid} | GT={gt_psd['count']}, V4={v4_psd['count']} "
            f"| PQ(V4 vs GT)={pq_gt_v4:.3f} | V4 D50 Err={rec['v4_D50_err_gt']}% ({v4_post_sec:.2f}s)"
        )

## Step 6: Write Incremental Drive Report Builder (`report_builder.py`)

In [ ]:
%%writefile /content/report_builder.py
import datetime, glob, json, os, platform, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def calc_r2_rmse(y_true: np.ndarray, y_pred: np.ndarray):
    mask = (~np.isnan(y_true)) & (~np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]
    if len(yt) < 2:
        return np.nan, np.nan
    rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    r2 = 1.0 - ss_res / max(ss_tot, 1e-9)
    return r2, rmse


def df_to_md_table(sub_df: pd.DataFrame) -> str:
    cols = list(sub_df.columns)
    header = '| ' + ' | '.join(cols) + ' |'
    sep = '| ' + ' | '.join([':---'] + ['---:'] * (len(cols) - 1)) + ' |'
    lines = [header, sep]
    for _, row in sub_df.iterrows():
        vals = ['N/A' if pd.isna(row[c]) else str(row[c]) for c in cols]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)


def rebuild_drive_report(drive_root: str, total_dataset_size: int):
    state_unet = sorted(glob.glob(os.path.join(drive_root, 'state', 'unet', '*.json')))
    state_cpsam = sorted(glob.glob(os.path.join(drive_root, 'state', 'cpsam', '*.json')))

    records = []
    for p in state_unet + state_cpsam:
        try:
            with open(p) as f:
                records.append(json.load(f))
        except Exception:
            pass

    if not records:
        return

    df = pd.DataFrame(records)
    csv_path = os.path.join(drive_root, 'full_dataset_benchmark.csv')
    tmp_csv = csv_path + '.tmp'
    df.to_csv(tmp_csv, index=False)
    os.replace(tmp_csv, csv_path)

    agg_rows = []
    clutter_rows = []
    for family, arch_name in [('unet', 'IG2 U-Net (25.3 MB)'), ('cpsam', 'IG2 Cellpose-SAM (~588 MB)')]:
        sub = df[df['family'] == family]
        if len(sub) == 0:
            continue
        sub_gt = sub[sub['has_gt'] == True]
        sub_clutter = sub[sub['has_gt'] == False]

        if len(sub_clutter) > 0:
            clutter_rows.append({
                'Model Family': arch_name,
                'Negative / Clutter Tiles': len(sub_clutter),
                'PT False-Positive Grains (Mean)': round(sub_clutter['pt_grains'].mean(), 2),
                'ONNX+CP False-Positive Grains (Mean)': round(sub_clutter['cp_grains'].mean(), 2),
                'BrowserV4 False-Positive Grains (Mean)': round(sub_clutter['v4_grains'].mean(), 2),
            })

        if len(sub_gt) == 0:
            continue

        r2_pt, rmse_pt = calc_r2_rmse(sub_gt['gt_D50'].astype(float).values, sub_gt['pt_D50'].astype(float).values)
        agg_rows.append({
            'Evaluated Model / Pipeline': f'Original PyTorch — {arch_name}',
            'Done / Total': f'{len(sub)} / {total_dataset_size} ({len(sub_gt)} GT)',
            'Mean PQ vs PT': '1.000 (Ref)',
            'Mean PQ vs Human GT': f"{sub_gt['pq_pt_vs_gt'].mean():.3f} ± {sub_gt['pq_pt_vs_gt'].std():.3f}",
            'D50 MAPE vs PT (%)': '0.00%',
            'D16 / D50 / D84 MAPE vs GT (%)': (
                f"{sub_gt['pt_D16_err_gt'].dropna().mean():.2f}% / "
                f"{sub_gt['pt_D50_err_gt'].dropna().mean():.2f}% / "
                f"{sub_gt['pt_D84_err_gt'].dropna().mean():.2f}%"
            ),
            'D50 RMSE vs GT (px) / R2': f'{rmse_pt:.2f} px / {r2_pt:.3f}',
            'Mean Time (s/img)': f"{sub['pt_sec'].mean():.2f}s",
        })

        r2_cp, rmse_cp = calc_r2_rmse(sub_gt['gt_D50'].astype(float).values, sub_gt['cp_D50'].astype(float).values)
        agg_rows.append({
            'Evaluated Model / Pipeline': f'ONNX + Cellpose Dynamics — {arch_name}',
            'Done / Total': f'{len(sub)} / {total_dataset_size} ({len(sub_gt)} GT)',
            'Mean PQ vs PT': f"{sub['pq_cp_vs_pt'].mean():.3f} ± {sub['pq_cp_vs_pt'].std():.3f}",
            'Mean PQ vs Human GT': f"{sub_gt['pq_cp_vs_gt'].mean():.3f} ± {sub_gt['pq_cp_vs_gt'].std():.3f}",
            'D50 MAPE vs PT (%)': f"{sub['cp_D50_err_pt'].dropna().mean():.2f}% (Med: {sub['cp_D50_err_pt'].dropna().median():.2f}%)",
            'D16 / D50 / D84 MAPE vs GT (%)': (
                f"{sub_gt['cp_D16_err_gt'].dropna().mean():.2f}% / "
                f"{sub_gt['cp_D50_err_gt'].dropna().mean():.2f}% / "
                f"{sub_gt['cp_D84_err_gt'].dropna().mean():.2f}%"
            ),
            'D50 RMSE vs GT (px) / R2': f'{rmse_cp:.2f} px / {r2_cp:.3f}',
            'Mean Time (s/img)': f"{(sub['onnx_inf_sec'] + sub['cp_post_sec']).mean():.2f}s",
        })

        r2_v4, rmse_v4 = calc_r2_rmse(sub_gt['gt_D50'].astype(float).values, sub_gt['v4_D50'].astype(float).values)
        agg_rows.append({
            'Evaluated Model / Pipeline': f'ONNX + Browser Engine v4 (F5+F3) — {arch_name}',
            'Done / Total': f'{len(sub)} / {total_dataset_size} ({len(sub_gt)} GT)',
            'Mean PQ vs PT': f"{sub['pq_v4_vs_pt'].mean():.3f} (vs CP: {sub['pq_v4_vs_cp'].mean():.3f})",
            'Mean PQ vs Human GT': f"{sub_gt['pq_v4_vs_gt'].mean():.3f} ± {sub_gt['pq_v4_vs_gt'].std():.3f}",
            'D50 MAPE vs PT (%)': f"{sub['v4_D50_err_pt'].dropna().mean():.2f}% (Med: {sub['v4_D50_err_pt'].dropna().median():.2f}%)",
            'D16 / D50 / D84 MAPE vs GT (%)': (
                f"{sub_gt['v4_D16_err_gt'].dropna().mean():.2f}% / "
                f"{sub_gt['v4_D50_err_gt'].dropna().mean():.2f}% / "
                f"{sub_gt['v4_D84_err_gt'].dropna().mean():.2f}%"
            ),
            'D50 RMSE vs GT (px) / R2': f'{rmse_v4:.2f} px / {r2_v4:.3f}',
            'Mean Time (s/img)': f"ONNX {sub['onnx_inf_sec'].mean():.2f}s + Post {sub['v4_post_sec'].mean():.2f}s",
        })

    df_agg = pd.DataFrame(agg_rows)

    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), dpi=135)
    for family, label, marker, color in [
        ('unet', 'IG2 U-Net FP32 (25.3 MB)', 'o', '#1f77b4'),
        ('cpsam', 'IG2 Cellpose-SAM FP16 (~588 MB)', 's', '#d62728'),
    ]:
        sub = df[df['family'] == family]
        if len(sub) > 0:
            axes[0].scatter(sub['pt_D50'], sub['v4_D50'], label=f'{label} (BrowserV4)', marker=marker, s=36, color=color, alpha=0.75)
            sub_gt = sub[sub['has_gt'] == True]
            if len(sub_gt) > 0:
                axes[1].scatter(sub_gt['gt_D50'], sub_gt['cp_D50'], label=f'{label} (ONNX+CP)', marker=marker, s=34, color=color, alpha=0.40)
                axes[1].scatter(sub_gt['gt_D50'], sub_gt['v4_D50'], label=f'{label} (BrowserV4)', marker='^', s=42, color=color, alpha=0.85)

    for ax, xlabel, ylabel, title in [
        (axes[0], 'Original PyTorch D50 (px)', 'ONNX + Browser Engine v4 D50 (px)', 'A. Browser Engine v4 vs. PyTorch D50 Parity'),
        (axes[1], 'Human Ground-Truth D50 (px)', 'Predicted D50 (px)', 'B. ONNX Models vs. Human Ground Truth (D50)'),
    ]:
        lims = [5, max(ax.get_xlim()[1], ax.get_ylim()[1], 20) * 1.08]
        ax.plot(lims, lims, 'k-', lw=1.5, label='1:1 Exact Parity')
        ax.fill_between(lims, [x * 0.90 for x in lims], [x * 1.10 for x in lims], color='gray', alpha=0.12, label='±10% Band')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left', fontsize=8.0)

    gt_only = df[df['has_gt'] == True]
    if len(gt_only) > 0:
        box_data = []
        box_labels = []
        for family, short_name in [('unet', 'U-Net'), ('cpsam', 'CP-SAM')]:
            sub_g = gt_only[gt_only['family'] == family]
            if len(sub_g) > 0:
                box_data.extend([
                    sub_g['pq_pt_vs_gt'].dropna().values,
                    sub_g['pq_cp_vs_gt'].dropna().values,
                    sub_g['pq_v4_vs_gt'].dropna().values,
                ])
                box_labels.extend([f'{short_name}\nPyTorch', f'{short_name}\nONNX+CP', f'{short_name}\nBrowserV4'])
        if box_data:
            axes[2].boxplot(box_data, labels=box_labels, patch_artist=True)
            axes[2].set_ylim(0, 1.0)
            axes[2].grid(True, axis='y', alpha=0.3)
            axes[2].set_ylabel('Panoptic Quality (PQ @ IoU ≥ 0.5 vs Human GT)')
            axes[2].set_title(f'C. Panoptic Quality Distribution (n={len(gt_only)//2} GT Images)')

    plt.tight_layout()
    agg_fig_path = os.path.join(drive_root, 'figures', 'aggregate_d50_parity.png')
    tmp_agg_fig = agg_fig_path + '.tmp.png'
    plt.savefig(tmp_agg_fig, bbox_inches='tight')
    plt.close(fig)
    os.replace(tmp_agg_fig, agg_fig_path)

    now_utc = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
    gpu_desc = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'

    md_lines = [
        '# ImageGrains 2.0 — Complete Ground-Truth Dataset Validation Report',
        '',
        f'- **Last Updated:** {now_utc}',
        f'- **Progress:** U-Net `{len(state_unet)} / {total_dataset_size}` | Cellpose-SAM `{len(state_cpsam)} / {total_dataset_size}`',
        f'- **Compute Hardware:** `{gpu_desc}` | Python {platform.python_version()} | PyTorch {torch.__version__}',
        '- **Hugging Face ONNX Repository:** [`inYourOwnBrowser/imagegrains-2.0-onnx`](https://huggingface.co/inYourOwnBrowser/imagegrains-2.0-onnx)',
        '',
        '---',
        '',
        '## 1. Aggregate Benchmark Metrics (All Grain-Bearing Ground-Truth Images)',
        '',
        'All morphometric and PSD metrics are computed on **border-excluded interior grains** (`margin = 2 px`) using **area-weighted cumulative distributions** ($w_i \\propto A_i$).',
        '',
        df_to_md_table(df_agg) if len(df_agg) > 0 else '_No completed images yet._',
        '',
        '![Aggregate Parity and Panoptic Quality](figures/aggregate_d50_parity.png)',
        '',
    ]

    if clutter_rows:
        md_lines.extend([
            '## 2. Non-Grain Clutter / Negative Control Tiles (`GT Grains = 0`)',
            '',
            df_to_md_table(pd.DataFrame(clutter_rows)),
            '',
        ])

    split_rows = []
    for family, arch_name in [('unet', 'IG2 U-Net (25.3 MB)'), ('cpsam', 'IG2 Cellpose-SAM (~588 MB)')]:
        sub_gt = df[(df['family'] == family) & (df['has_gt'] == True)]
        for sp in ['test', 'train']:
            sp_df = sub_gt[sub_gt['split'] == sp]
            if len(sp_df) == 0:
                continue
            r2_v4, rmse_v4 = calc_r2_rmse(sp_df['gt_D50'].astype(float).values, sp_df['v4_D50'].astype(float).values)
            split_rows.append({
                'Model Family': arch_name,
                'Dataset Split': sp.upper(),
                'Images': len(sp_df),
                'PyTorch D50 MAPE (%)': f"{sp_df['pt_D50_err_gt'].dropna().mean():.2f}%",
                'ONNX+CP D50 MAPE (%)': f"{sp_df['cp_D50_err_gt'].dropna().mean():.2f}%",
                'BrowserV4 D50 MAPE (%)': f"{sp_df['v4_D50_err_gt'].dropna().mean():.2f}%",
                'BrowserV4 D50 RMSE / R2': f'{rmse_v4:.2f} px / {r2_v4:.3f}',
                'BrowserV4 PQ vs GT': f"{sp_df['pq_v4_vs_gt'].mean():.3f}",
            })
    if split_rows:
        md_lines.extend([
            '## 3. Breakdown by Official Held-Out `TEST` Split vs. `TRAIN` Split',
            '',
            df_to_md_table(pd.DataFrame(split_rows)),
            '',
        ])

    md_lines.extend([
        '## 4. Full Per-Image Results Table',
        '',
        'Full per-image metrics are saved in `full_dataset_benchmark.csv` and individual JSON records in `state/unet/` and `state/cpsam/`.',
        '',
        df_to_md_table(df[[
            'model', 'split', 'site', 'uid', 'shape',
            'gt_grains', 'pt_grains', 'cp_grains', 'v4_grains',
            'pq_v4_vs_cp', 'pq_pt_vs_gt', 'pq_v4_vs_gt',
            'gt_D50', 'pt_D50', 'v4_D50', 'v4_D50_err_gt', 'v4_D50_err_pt'
        ]]),
    ])

    md_path = os.path.join(drive_root, 'full_dataset_validation_report.md')
    tmp_md = md_path + '.tmp'
    with open(tmp_md, 'w', encoding='utf-8') as f:
        f.write('\n'.join(md_lines))
    os.replace(tmp_md, md_path)

## Step 7: Run Resumable Batched Full-Dataset Benchmark
Processes `BATCH_SIZE = 20` images per batch and checkpoints reports to Google Drive after every batch. Safe to interrupt and re-run at any time.

In [ ]:
import glob, json, os, subprocess, sys, time
sys.path.insert(0, '/content')
from eval_core import finalize_batch_to_drive
from report_builder import rebuild_drive_report

DRIVE_ROOT = '/content/drive/MyDrive/imagegrains_full_validation'
MANIFEST_PATH = '/content/test_images/full_dataset_manifest.json'

BATCH_SIZE = 20          # Number of images processed per batch before checkpointing reports to Drive
SAVE_ALL_FIGURES = True  # Saves 5-panel comparison PNG for every image to Drive/figures/
RUN_UNET = True          # Set False if you only want to run Cellpose-SAM
RUN_CPSAM = True         # Set False if you only want to run U-Net

with open(MANIFEST_PATH) as f:
    all_records = json.load(f)

total_n = len(all_records)

def get_completed_uids(family: str) -> set[str]:
    state_dir = os.path.join(DRIVE_ROOT, 'state', family)
    return {
        os.path.splitext(fn)[0]
        for fn in os.listdir(state_dir)
        if fn.endswith('.json') and not fn.endswith('.tmp')
    }

def build_env(venv_path: str) -> dict:
    env = os.environ.copy()
    nvidia_libs = glob.glob(f'{venv_path}/lib/python3.11/site-packages/nvidia/*/lib')
    env['LD_LIBRARY_PATH'] = ':'.join(nvidia_libs + [env.get('LD_LIBRARY_PATH', '')])
    env['PYTHONUNBUFFERED'] = '1'
    return env

def run_worker_cmd(cmd: list[str], env: dict):
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail_lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        tail_lines.append(line)
        if len(tail_lines) > 60:
            tail_lines.pop(0)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Worker failed with exit code {ret}. Traceback / output:\n{''.join(tail_lines)}")

env_cp3 = build_env('/content/venv_cp3')
env_cp4 = build_env('/content/venv_cp4')

unet_done = get_completed_uids('unet')
cpsam_done = get_completed_uids('cpsam')
print('=== RESUMABLE FULL-DATASET BENCHMARK ===')
print(f'Total unique ground-truth examples: {total_n}')
print(f'Already completed on Drive -> U-Net: {len(unet_done)}/{total_n} | Cellpose-SAM: {len(cpsam_done)}/{total_n}')

batch_num = 0
while True:
    unet_done = get_completed_uids('unet') if RUN_UNET else {r['uid'] for r in all_records}
    cpsam_done = get_completed_uids('cpsam') if RUN_CPSAM else {r['uid'] for r in all_records}

    pending_records = [
        r for r in all_records
        if (RUN_UNET and r['uid'] not in unet_done) or (RUN_CPSAM and r['uid'] not in cpsam_done)
    ]

    if not pending_records:
        print('\nAll ground-truth examples have been completed!')
        break

    batch_num += 1
    current_batch = pending_records[:BATCH_SIZE]
    batch_file = '/content/current_batch.json'
    with open(batch_file, 'w') as f:
        json.dump(current_batch, f, indent=2)

    print(f'\n--- Batch {batch_num} ({len(current_batch)} images | {len(pending_records)} remaining overall) ---')
    t_batch = time.perf_counter()

    if RUN_UNET:
        run_worker_cmd([
            '/content/venv_cp3/bin/python', '/content/infer_unet.py',
            '--batch-file', batch_file,
            '--state-dir', os.path.join(DRIVE_ROOT, 'state', 'unet'),
            '--tmp-dir', '/content/batch_tmp/unet',
        ], env=env_cp3)
        finalize_batch_to_drive(
            current_batch,
            family='unet',
            model_label='IG2 U-Net FP32 (25.3 MB)',
            tmp_dir='/content/batch_tmp/unet',
            state_dir=os.path.join(DRIVE_ROOT, 'state', 'unet'),
            fig_dir=os.path.join(DRIVE_ROOT, 'figures'),
            save_figures=SAVE_ALL_FIGURES,
        )

    if RUN_CPSAM:
        run_worker_cmd([
            '/content/venv_cp4/bin/python', '/content/infer_cpsam.py',
            '--batch-file', batch_file,
            '--state-dir', os.path.join(DRIVE_ROOT, 'state', 'cpsam'),
            '--tmp-dir', '/content/batch_tmp/cpsam',
        ], env=env_cp4)
        finalize_batch_to_drive(
            current_batch,
            family='cpsam',
            model_label='IG2 Cellpose-SAM FP16 (~588 MB)',
            tmp_dir='/content/batch_tmp/cpsam',
            state_dir=os.path.join(DRIVE_ROOT, 'state', 'cpsam'),
            fig_dir=os.path.join(DRIVE_ROOT, 'figures'),
            save_figures=SAVE_ALL_FIGURES,
        )

    rebuild_drive_report(DRIVE_ROOT, total_n)
    dt_batch = time.perf_counter() - t_batch
    print(
        f'Batch {batch_num} checkpointed to Drive in {dt_batch:.1f}s | '
        f'Progress -> U-Net: {len(get_completed_uids("unet"))}/{total_n} | CP-SAM: {len(get_completed_uids("cpsam"))}/{total_n}'
    )

rebuild_drive_report(DRIVE_ROOT, total_n)
print(f'\nComplete! Full report saved at: {DRIVE_ROOT}/full_dataset_validation_report.md')

## Step 8: Save Production TypeScript Web Worker v4 (`grain_postprocess.worker.ts`) to Drive

In [ ]:
%%writefile /content/drive/MyDrive/imagegrains_full_validation/grain_postprocess.worker.ts
/**
 * Browser-Native Cellpose / ImageGrains 2.0 Post-Processing Engine (v4 — Ablation-Optimized)
 * Zero external dependencies — pure TypedArray math (Float32Array / Int32Array / Uint8Array).
 *
 * Implements the winning feature combination validated across 25 multi-site ground-truth images:
 *   1. Two-phase sub-pixel bilinear Euler flow integration (coarse stride + 1.0 px precision)
 *   2. 2D sink density histogram + 2-pass 3x3 morphological seed dilation
 *   3. 8-connected component seed labeling + stray boundary pixel snapping
 *   4. Coarse-to-fine Multigrid 9-point Jacobi heat-diffusion PDE (4x -> 2x -> 1x)
 *   5. F5: Flow-magnitude weighted QC (w = min(1, ||dP||/1.5)) + 3x3 center-sink exclusion
 *   6. F3: Two-peak divergence/sink split-on-failure recovery for merged touching clasts
 *   7. Border-grain exclusion & second-order central image moments (a-axis, b-axis, d_eq)
 */

export interface PostProcessOptions {
  niter?: number;
  cellprobThreshold?: number;
  flowThreshold?: number;
  minSize?: number;
  minSeedDensity?: number;
  excludeBorderMargin?: number;
}

export interface GrainMorphometrics {
  id: number;
  areaPx: number;
  aAxisPx: number;
  bAxisPx: number;
  dEqPx: number;
  centroidX: number;
  centroidY: number;
  touchesBorder: boolean;
}

function runJacobiSteps(
  mask: Uint8Array,
  T: Float32Array,
  Tnext: Float32Array,
  bh: number,
  bw: number,
  centerOffset: number,
  steps: number
): Float32Array {
  const pw = bw + 2;
  let cur = T;
  let nxt = Tnext;
  for (let it = 0; it < steps; it++) {
    cur[centerOffset] += 1.0;
    for (let py = 1; py <= bh; py++) {
      const r = py * pw;
      const rUp = r - pw;
      const rDn = r + pw;
      for (let px = 1; px <= bw; px++) {
        const idx = r + px;
        if (mask[idx]) {
          nxt[idx] =
            (cur[rUp + px - 1] +
              cur[rUp + px] +
              cur[rUp + px + 1] +
              cur[r + px - 1] +
              cur[idx] +
              cur[r + px + 1] +
              cur[rDn + px - 1] +
              cur[rDn + px] +
              cur[rDn + px + 1]) *
            (1.0 / 9.0);
        } else {
          nxt[idx] = 0.0;
        }
      }
    }
    const tmp = cur;
    cur = nxt;
    nxt = tmp;
  }
  return cur;
}

function multigridJacobiHeat(
  subMask: Uint8Array,
  bh: number,
  bw: number,
  cy: number,
  cx: number
): Float32Array {
  const pw = bw + 2;
  const ph = bh + 2;
  const T = new Float32Array(pw * ph);
  const Tnext = new Float32Array(pw * ph);
  const maxDim = Math.max(bh, bw);

  let nFine = Math.max(50, 2 * (bh + bw));
  if (maxDim >= 96) {
    const scale = 4;
    const h4 = Math.ceil(bh / scale);
    const w4 = Math.ceil(bw / scale);
    const pw4 = w4 + 2;
    const ph4 = h4 + 2;
    const m4 = new Uint8Array(pw4 * ph4);
    for (let y4 = 0; y4 < h4; y4++) {
      const sy = Math.min(y4 * scale, bh - 1) + 1;
      for (let x4 = 0; x4 < w4; x4++) {
        const sx = Math.min(x4 * scale, bw - 1) + 1;
        m4[(y4 + 1) * pw4 + (x4 + 1)] = subMask[sy * pw + sx];
      }
    }
    const cy4 = Math.min(Math.floor(cy / scale), h4 - 1) + 1;
    const cx4 = Math.min(Math.floor(cx / scale), w4 - 1) + 1;
    const T4 = runJacobiSteps(
      m4,
      new Float32Array(pw4 * ph4),
      new Float32Array(pw4 * ph4),
      h4,
      w4,
      cy4 * pw4 + cx4,
      Math.max(70, 2 * (h4 + w4))
    );
    for (let dy = 0; dy < bh; dy++) {
      const y4 = Math.min(Math.floor(dy / scale), h4 - 1) + 1;
      for (let dx = 0; dx < bw; dx++) {
        const idx = (dy + 1) * pw + (dx + 1);
        if (subMask[idx]) {
          const x4 = Math.min(Math.floor(dx / scale), w4 - 1) + 1;
          T[idx] = T4[y4 * pw4 + x4];
        }
      }
    }
    nFine = Math.max(70, bh + bw);
  } else if (maxDim >= 48) {
    const scale = 2;
    const h2 = Math.ceil(bh / scale);
    const w2 = Math.ceil(bw / scale);
    const pw2 = w2 + 2;
    const ph2 = h2 + 2;
    const m2 = new Uint8Array(pw2 * ph2);
    for (let y2 = 0; y2 < h2; y2++) {
      const sy = Math.min(y2 * scale, bh - 1) + 1;
      for (let x2 = 0; x2 < w2; x2++) {
        const sx = Math.min(x2 * scale, bw - 1) + 1;
        m2[(y2 + 1) * pw2 + (x2 + 1)] = subMask[sy * pw + sx];
      }
    }
    const cy2 = Math.min(Math.floor(cy / scale), h2 - 1) + 1;
    const cx2 = Math.min(Math.floor(cx / scale), w2 - 1) + 1;
    const T2 = runJacobiSteps(
      m2,
      new Float32Array(pw2 * ph2),
      new Float32Array(pw2 * ph2),
      h2,
      w2,
      cy2 * pw2 + cx2,
      Math.max(70, 2 * (h2 + w2))
    );
    for (let dy = 0; dy < bh; dy++) {
      const y2 = Math.min(Math.floor(dy / scale), h2 - 1) + 1;
      for (let dx = 0; dx < bw; dx++) {
        const idx = (dy + 1) * pw + (dx + 1);
        if (subMask[idx]) {
          const x2 = Math.min(Math.floor(dx / scale), w2 - 1) + 1;
          T[idx] = T2[y2 * pw2 + x2];
        }
      }
    }
    nFine = Math.max(70, bh + bw);
  }

  return runJacobiSteps(subMask, T, Tnext, bh, bw, (cy + 1) * pw + (cx + 1), nFine);
}

function evaluateGrainFlowQcV4(
  subMask: Uint8Array,
  bh: number,
  bw: number,
  y0: number,
  x0: number,
  width: number,
  uy: Float32Array,
  ux: Float32Array,
  rawMag: Float32Array,
  flowThresh: number
): boolean {
  const pw = bw + 2;
  let area = 0;
  let sumY = 0;
  let sumX = 0;
  for (let dy = 0; dy < bh; dy++) {
    const r = (dy + 1) * pw + 1;
    for (let dx = 0; dx < bw; dx++) {
      if (subMask[r + dx]) {
        area++;
        sumY += dy;
        sumX += dx;
      }
    }
  }
  if (area === 0) return false;

  const meanY = sumY / area;
  const meanX = sumX / area;
  let bestDist = Infinity;
  let cy = 0;
  let cx = 0;
  for (let dy = 0; dy < bh; dy++) {
    const r = (dy + 1) * pw + 1;
    for (let dx = 0; dx < bw; dx++) {
      if (subMask[r + dx]) {
        const d2 = (dy - meanY) ** 2 + (dx - meanX) ** 2;
        if (d2 < bestDist) {
          bestDist = d2;
          cy = dy;
          cx = dx;
        }
      }
    }
  }

  const T = multigridJacobiHeat(subMask, bh, bw, cy, cx);

  // F5: Magnitude-weighted flow QC + 3x3 center-sink exclusion
  let weightedErrSum = 0.0;
  let weightSum = 0.0;

  for (let dy = 0; dy < bh; dy++) {
    const pySub = dy + 1;
    const r = pySub * pw;
    const imgRow = (y0 + dy) * width + x0;
    const dCy = dy - cy;
    for (let dx = 0; dx < bw; dx++) {
      const pxSub = dx + 1;
      const idx = r + pxSub;
      if (!subMask[idx]) continue;

      const dist2 = dCy * dCy + (dx - cx) * (dx - cx);
      if (dist2 <= 2) continue;

      const imgIdx = imgRow + dx;
      const wMag = Math.min(1.0, rawMag[imgIdx] / 1.5);
      if (wMag <= 0.0) continue;

      const gy = T[idx + pw] - T[idx - pw];
      const gx = T[idx + 1] - T[idx - 1];
      const gmag = Math.hypot(gy, gx) + 1e-6;
      const idUy = gy / gmag;
      const idUx = gx / gmag;

      const errY = 5.0 * (uy[imgIdx] - idUy);
      const errX = 5.0 * (ux[imgIdx] - idUx);
      weightedErrSum += (errY * errY + errX * errX) * wMag;
      weightSum += wMag;
    }
  }

  const flowErr = weightedErrSum / Math.max(weightSum, 1e-6);
  return flowErr / 10.0 <= flowThresh;
}

function trySplitDumbbellV4(
  subMask: Uint8Array,
  bh: number,
  bw: number,
  y0: number,
  x0: number,
  width: number,
  uy: Float32Array,
  ux: Float32Array,
  rawMag: Float32Array,
  sinkHist: Int32Array,
  minSize: number,
  flowThresh: number
): Uint8Array[] {
  const pw = bw + 2;
  const subN = bh * bw;
  const score = new Float32Array(subN).fill(-1.0);

  // Score interior mask pixels by convergence density + inward flow divergence
  for (let dy = 0; dy < bh; dy++) {
    const r = (dy + 1) * pw + 1;
    const iy = y0 + dy;
    const iyUp = dy > 0 ? iy - 1 : iy;
    const iyDn = dy + 1 < bh ? iy + 1 : iy;
    const dySpan = iyDn - iyUp || 1;
    for (let dx = 0; dx < bw; dx++) {
      if (!subMask[r + dx]) continue;
      const ix = x0 + dx;
      const ixLt = dx > 0 ? ix - 1 : ix;
      const ixRt = dx + 1 < bw ? ix + 1 : ix;
      const dxSpan = ixRt - ixLt || 1;

      const duY = (uy[iyDn * width + ix] - uy[iyUp * width + ix]) / dySpan;
      const duX = (ux[iy * width + ixRt] - ux[iy * width + ixLt]) / dxSpan;
      const inwardDiv = Math.max(0.0, -(duY + duX));
      score[dy * bw + dx] = sinkHist[iy * width + ix] + 2.0 * inwardDiv;
    }
  }

  let p1 = -1;
  let best1 = -1.0;
  for (let i = 0; i < subN; i++) {
    if (score[i] > best1) {
      best1 = score[i];
      p1 = i;
    }
  }
  if (p1 < 0) return [];
  const y1 = (p1 / bw) | 0;
  const x1 = p1 % bw;

  let p2 = -1;
  let best2 = 0.0;
  for (let dy = 0; dy < bh; dy++) {
    for (let dx = 0; dx < bw; dx++) {
      if ((dy - y1) ** 2 + (dx - x1) ** 2 < 36) continue;
      const s = score[dy * bw + dx];
      if (s > best2) {
        best2 = s;
        p2 = dy * bw + dx;
      }
    }
  }
  if (p2 < 0) return [];
  const y2 = (p2 / bw) | 0;
  const x2 = p2 % bw;

  // Competitive 2-seed 8-connected BFS partition
  const part = new Uint8Array(subN);
  const queue = new Int32Array(subN);
  let head = 0;
  let tail = 0;
  part[p1] = 1;
  part[p2] = 2;
  queue[tail++] = p1;
  queue[tail++] = p2;

  while (head < tail) {
    const curr = queue[head++];
    const cid = part[curr];
    const cy = (curr / bw) | 0;
    const cx = curr % bw;
    const yLo = cy > 0 ? cy - 1 : 0;
    const yHi = cy + 1 < bh ? cy + 1 : bh - 1;
    const xLo = cx > 0 ? cx - 1 : 0;
    const xHi = cx + 1 < bw ? cx + 1 : bw - 1;
    for (let ny = yLo; ny <= yHi; ny++) {
      for (let nx = xLo; nx <= xHi; nx++) {
        const ni = ny * bw + nx;
        if (part[ni] === 0 && subMask[(ny + 1) * pw + (nx + 1)]) {
          part[ni] = cid;
          queue[tail++] = ni;
        }
      }
    }
  }

  // Nearest-seed fallback for any disconnected pixels inside subMask
  for (let dy = 0; dy < bh; dy++) {
    for (let dx = 0; dx < bw; dx++) {
      const ni = dy * bw + dx;
      if (part[ni] === 0 && subMask[(dy + 1) * pw + (dx + 1)]) {
        const d1 = (dy - y1) ** 2 + (dx - x1) ** 2;
        const d2 = (dy - y2) ** 2 + (dx - x2) ** 2;
        part[ni] = d1 <= d2 ? 1 : 2;
      }
    }
  }

  const validChildren: Uint8Array[] = [];
  for (const cid of [1, 2]) {
    const childMask = new Uint8Array(pw * (bh + 2));
    let childArea = 0;
    for (let dy = 0; dy < bh; dy++) {
      const dstRow = (dy + 1) * pw + 1;
      for (let dx = 0; dx < bw; dx++) {
        if (part[dy * bw + dx] === cid) {
          childMask[dstRow + dx] = 1;
          childArea++;
        }
      }
    }
    if (childArea < minSize) continue;
    if (evaluateGrainFlowQcV4(childMask, bh, bw, y0, x0, width, uy, ux, rawMag, flowThresh)) {
      validChildren.push(childMask);
    }
  }
  return validChildren;
}

export function flowsToMasksBrowser(
  flowY: Float32Array,
  flowX: Float32Array,
  cellprob: Float32Array,
  width: number,
  height: number,
  opts: PostProcessOptions = {}
): { masks: Int32Array; grains: GrainMorphometrics[] } {
  const niter = opts.niter ?? 250;
  const cellprobThresh = opts.cellprobThreshold ?? 0.0;
  const flowThresh = opts.flowThreshold ?? 0.45;
  const minSize = opts.minSize ?? 30;
  const minSeedDensity = opts.minSeedDensity ?? 2;
  const borderMargin = opts.excludeBorderMargin ?? 2;

  const N = width * height;
  const rawMag = new Float32Array(N);
  const uy = new Float32Array(N);
  const ux = new Float32Array(N);

  // 1. Collect foreground pixels & unit-normalize flow vectors
  const fgIdx = new Int32Array(N);
  let fgCount = 0;
  for (let i = 0; i < N; i++) {
    if (cellprob[i] > cellprobThresh) {
      fgIdx[fgCount++] = i;
      const fy = flowY[i];
      const fx = flowX[i];
      const mag = Math.hypot(fy, fx);
      rawMag[i] = mag;
      const invMag = 1.0 / (mag + 1e-6);
      uy[i] = fy * invMag;
      ux[i] = fx * invMag;
    }
  }

  if (fgCount === 0) {
    return { masks: new Int32Array(N), grains: [] };
  }

  // 2. Two-phase sub-pixel bilinear Euler trajectory integration
  const py = new Float32Array(fgCount);
  const px = new Float32Array(fgCount);
  for (let k = 0; k < fgCount; k++) {
    const idx = fgIdx[k];
    py[k] = (idx / width) | 0;
    px[k] = idx % width;
  }

  const wMax = width - 1;
  const hMax = height - 1;
  const coarseSteps = Math.min(100, (niter / 2) | 0);
  const coarseStride = Math.max(1.0, Math.max(width, height) / 256.0);

  for (let step = 0; step < niter; step++) {
    const stepScale = step < coarseSteps ? coarseStride : 1.0;
    for (let k = 0; k < fgCount; k++) {
      const y = py[k];
      const x = px[k];
      const y0 = y | 0;
      const x0 = x | 0;
      const y1 = y0 < hMax ? y0 + 1 : hMax;
      const x1 = x0 < wMax ? x0 + 1 : wMax;
      const wy = y - y0;
      const wx = x - x0;
      const wy1 = 1.0 - wy;
      const wx1 = 1.0 - wx;

      const w00 = wy1 * wx1;
      const w01 = wy1 * wx;
      const w10 = wy * wx1;
      const w11 = wy * wx;

      const row0 = y0 * width;
      const row1 = y1 * width;
      const i00 = row0 + x0;
      const i01 = row0 + x1;
      const i10 = row1 + x0;
      const i11 = row1 + x1;

      const dy = w00 * uy[i00] + w01 * uy[i01] + w10 * uy[i10] + w11 * uy[i11];
      const dx = w00 * ux[i00] + w01 * ux[i01] + w10 * ux[i10] + w11 * ux[i11];

      let ny = y + stepScale * dy;
      let nx = x + stepScale * dx;
      if (ny < 0) ny = 0;
      else if (ny > hMax) ny = hMax;
      if (nx < 0) nx = 0;
      else if (nx > wMax) nx = wMax;
      py[k] = ny;
      px[k] = nx;
    }
  }

  // 3. 2D Convergence Histogram + 2-Pass 3x3 Seed Dilation
  const sinkHist = new Int32Array(N);
  const endIdx = new Int32Array(fgCount);
  let hasDenseSeed = false;

  for (let k = 0; k < fgCount; k++) {
    let sy = Math.round(py[k]);
    let sx = Math.round(px[k]);
    if (sy < 0) sy = 0;
    else if (sy > hMax) sy = hMax;
    if (sx < 0) sx = 0;
    else if (sx > wMax) sx = wMax;
    const e = sy * width + sx;
    endIdx[k] = e;
    const c = ++sinkHist[e];
    if (c >= minSeedDensity) hasDenseSeed = true;
  }

  const thresh = hasDenseSeed ? minSeedDensity : 1;
  let seedMask = new Uint8Array(N);
  for (let i = 0; i < N; i++) {
    if (sinkHist[i] >= thresh) seedMask[i] = 1;
  }

  for (let pass = 0; pass < 2; pass++) {
    const next = new Uint8Array(N);
    for (let y = 0; y < height; y++) {
      const yLo = y > 0 ? y - 1 : 0;
      const yHi = y < hMax ? y + 1 : hMax;
      for (let x = 0; x < width; x++) {
        if (seedMask[y * width + x]) {
          next[y * width + x] = 1;
          continue;
        }
        const xLo = x > 0 ? x - 1 : 0;
        const xHi = x < wMax ? x + 1 : wMax;
        let hit = 0;
        for (let ny = yLo; ny <= yHi && !hit; ny++) {
          const r = ny * width;
          for (let nx = xLo; nx <= xHi; nx++) {
            if (seedMask[r + nx]) {
              hit = 1;
              break;
            }
          }
        }
        next[y * width + x] = hit;
      }
    }
    seedMask = next;
  }

  // 4. 8-Connected Component Labeling on Dilated Seeds (BFS Queue)
  const seedLabels = new Int32Array(N);
  const queue = new Int32Array(N);
  let nSeeds = 0;

  for (let i = 0; i < N; i++) {
    if (!seedMask[i] || seedLabels[i] !== 0) continue;
    nSeeds++;
    seedLabels[i] = nSeeds;
    let head = 0;
    let tail = 0;
    queue[tail++] = i;

    while (head < tail) {
      const curr = queue[head++];
      const cy = (curr / width) | 0;
      const cx = curr % width;
      const yLo = cy > 0 ? cy - 1 : 0;
      const yHi = cy < hMax ? cy + 1 : hMax;
      const xLo = cx > 0 ? cx - 1 : 0;
      const xHi = cx < wMax ? cx + 1 : wMax;

      for (let ny = yLo; ny <= yHi; ny++) {
        const r = ny * width;
        for (let nx = xLo; nx <= xHi; nx++) {
          const ni = r + nx;
          if (seedMask[ni] && seedLabels[ni] === 0) {
            seedLabels[ni] = nSeeds;
            queue[tail++] = ni;
          }
        }
      }
    }
  }

  if (nSeeds === 0) {
    return { masks: new Int32Array(N), grains: [] };
  }

  // 5. Assign Foreground Pixels + Snap Stray Boundary Pixels within 2 px
  const rawLabels = new Int32Array(N);
  const areaCounts = new Int32Array(nSeeds + 1);
  const yMin = new Int32Array(nSeeds + 1).fill(height);
  const yMax = new Int32Array(nSeeds + 1).fill(-1);
  const xMin = new Int32Array(nSeeds + 1).fill(width);
  const xMax = new Int32Array(nSeeds + 1).fill(-1);

  for (let k = 0; k < fgCount; k++) {
    const e = endIdx[k];
    let lbl = seedLabels[e];
    if (lbl === 0) {
      const ey = (e / width) | 0;
      const ex = e % width;
      const yLo = ey >= 2 ? ey - 2 : 0;
      const yHi = ey + 2 <= hMax ? ey + 2 : hMax;
      const xLo = ex >= 2 ? ex - 2 : 0;
      const xHi = ex + 2 <= wMax ? ex + 2 : wMax;
      for (let ny = yLo; ny <= yHi && lbl === 0; ny++) {
        const r = ny * width;
        for (let nx = xLo; nx <= xHi; nx++) {
          const cand = seedLabels[r + nx];
          if (cand > 0) {
            lbl = cand;
            break;
          }
        }
      }
    }
    if (lbl > 0) {
      const orig = fgIdx[k];
      const oy = (orig / width) | 0;
      const ox = orig % width;
      rawLabels[orig] = lbl;
      areaCounts[lbl]++;
      if (oy < yMin[lbl]) yMin[lbl] = oy;
      if (oy > yMax[lbl]) yMax[lbl] = oy;
      if (ox < xMin[lbl]) xMin[lbl] = ox;
      if (ox > xMax[lbl]) xMax[lbl] = ox;
    }
  }

  // 6. Multigrid Heat-Diffusion PDE + F5 Mag-Weighted QC + F3 Split-on-Failure Recovery
  const finalMasks = new Int32Array(N);
  let finalCount = 0;

  for (let mid = 1; mid <= nSeeds; mid++) {
    const area = areaCounts[mid];
    if (area < minSize) continue;

    const y0 = yMin[mid];
    const y1 = yMax[mid];
    const x0 = xMin[mid];
    const x1 = xMax[mid];
    const bh = y1 - y0 + 1;
    const bw = x1 - x0 + 1;
    const pw = bw + 2;
    const ph = bh + 2;

    const subMask = new Uint8Array(pw * ph);
    for (let dy = 0; dy < bh; dy++) {
      const srcRow = (y0 + dy) * width + x0;
      const dstRow = (dy + 1) * pw + 1;
      for (let dx = 0; dx < bw; dx++) {
        if (rawLabels[srcRow + dx] === mid) {
          subMask[dstRow + dx] = 1;
        }
      }
    }

    if (evaluateGrainFlowQcV4(subMask, bh, bw, y0, x0, width, uy, ux, rawMag, flowThresh)) {
      const id = ++finalCount;
      for (let dy = 0; dy < bh; dy++) {
        const srcRow = (dy + 1) * pw + 1;
        const dstRow = (y0 + dy) * width + x0;
        for (let dx = 0; dx < bw; dx++) {
          if (subMask[srcRow + dx] && finalMasks[dstRow + dx] === 0) {
            finalMasks[dstRow + dx] = id;
          }
        }
      }
    } else {
      const children = trySplitDumbbellV4(
        subMask,
        bh,
        bw,
        y0,
        x0,
        width,
        uy,
        ux,
        rawMag,
        sinkHist,
        minSize,
        flowThresh
      );
      for (const childMask of children) {
        const id = ++finalCount;
        for (let dy = 0; dy < bh; dy++) {
          const srcRow = (dy + 1) * pw + 1;
          const dstRow = (y0 + dy) * width + x0;
          for (let dx = 0; dx < bw; dx++) {
            if (childMask[srcRow + dx] && finalMasks[dstRow + dx] === 0) {
              finalMasks[dstRow + dx] = id;
            }
          }
        }
      }
    }
  }

  // 7. Compute Second-Order Central Moments (a-axis, b-axis, d_eq) & Border Flags
  const m00 = new Float64Array(finalCount + 1);
  const m10 = new Float64Array(finalCount + 1);
  const m01 = new Float64Array(finalCount + 1);
  const m20 = new Float64Array(finalCount + 1);
  const m02 = new Float64Array(finalCount + 1);
  const m11 = new Float64Array(finalCount + 1);
  const touchesBorder = new Uint8Array(finalCount + 1);

  for (let y = 0; y < height; y++) {
    const r = y * width;
    const isBorderY = y < borderMargin || y >= height - borderMargin;
    for (let x = 0; x < width; x++) {
      const id = finalMasks[r + x];
      if (id === 0) continue;
      m00[id] += 1;
      m10[id] += x;
      m01[id] += y;
      m20[id] += x * x;
      m02[id] += y * y;
      m11[id] += x * y;
      if (isBorderY || x < borderMargin || x >= width - borderMargin) {
        touchesBorder[id] = 1;
      }
    }
  }

  const grains: GrainMorphometrics[] = [];
  for (let id = 1; id <= finalCount; id++) {
    const area = m00[id];
    if (area < minSize) continue;
    const cx = m10[id] / area;
    const cy = m01[id] / area;
    const mu20 = m20[id] / area - cx * cx;
    const mu02 = m02[id] / area - cy * cy;
    const mu11 = m11[id] / area - cx * cy;

    const common = Math.hypot(mu20 - mu02, 2.0 * mu11);
    const l1 = Math.max(0.0, (mu20 + mu02 + common) * 0.5);
    const l2 = Math.max(0.0, (mu20 + mu02 - common) * 0.5);

    const aAxis = 4.0 * Math.sqrt(l1);
    const bAxis = 4.0 * Math.sqrt(l2);
    const dEq = Math.sqrt((4.0 * area) / Math.PI);

    grains.push({
      id,
      areaPx: area,
      aAxisPx: aAxis,
      bAxisPx: bAxis,
      dEqPx: dEq,
      centroidX: cx,
      centroidY: cy,
      touchesBorder: touchesBorder[id] === 1,
    });
  }

  return { masks: finalMasks, grains };
}

## Step 9: Display Final Aggregate Parity Plots & Split Summary Table

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

DRIVE_ROOT = '/content/drive/MyDrive/imagegrains_full_validation'
csv_path = os.path.join(DRIVE_ROOT, 'full_dataset_benchmark.csv')
fig_path = os.path.join(DRIVE_ROOT, 'figures', 'aggregate_d50_parity.png')

if os.path.exists(fig_path):
    display(Image(filename=fig_path))

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Total benchmarked model-image rows: {len(df)}')
    display(df.groupby(['model', 'split', 'has_gt'])[['pq_v4_vs_cp', 'pq_pt_vs_gt', 'pq_v4_vs_gt', 'pt_D50_err_gt', 'cp_D50_err_gt', 'v4_D50_err_gt']].mean().round(3))